In [ ]:
request!pip install --no-input streamlit pymongo dnspython python-dotenv bcrypt pyjwt

In [ ]:
!pip install reportlab

In [ ]:
!pip install streamlit-lottie --quiet


In [ ]:
%%writefile .env
# MongoDB Atlas connection
MONGO_URI=

# JWT secret
JWT_SECRET="CodeGenieAI_M3_Secret_2025"

# Hugging Face token (must be valid to load private models)
HUGGINGFACE_TOKEN=

# Optional: Provide specific model overrides (leave empty to use defaults in code)
PHI3_MODEL=""
GEMMA2B_MODEL=""

# SMTP settings (for sending OTP emails). If left empty, app shows OTP in UI for demo.
SMTP_HOST="smtp.gmail.com"
SMTP_PORT="587"
SMTP_USER="mukeshmugi1114@gmail.com"        # your_gmail@gmail.com
SMTP_PASS="cczs dfpl tths hewy"        # app password (create an app password in Google account)
EMAIL_FROM="CodeGenie <mukeshmugi1114@gmail.com>"

# Initial Admin User (Optional - set if you want an admin user created on first run when DB is empty)
ADMIN_INITIAL_USER="codegenie3@gmail.com" # REPLACE WITH YOUR DESIRED ADMIN EMAIL
ADMIN_INITIAL_PASS="infosyskijaiho" # REPLACE WITH A SECURE PASSWORD



APP_BASE_URL=" https://unshed-defiantly-sutton.ngrok-free.dev"
GOOGLE_CLIENT_ID="1014254344168-vvkkjne9a2c5rd37is3f8hcrapej8v7p.apps.googleusercontent.com"
GOOGLE_CLIENT_SECRET="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"
GITHUB_CLIENT_ID="..."
GITHUB_CLIENT_SECRET="..."
LINKEDIN_CLIENT_ID="..."
LINKEDIN_CLIENT_SECRET="..."


In [ ]:
import pymongo
from dotenv import load_dotenv
import os

load_dotenv()

MONGO_URI = os.getenv("MONGO_URI")

def get_db():
    try:
        client = pymongo.MongoClient(MONGO_URI)
        db = client["CodeGenieDB"]  # database name
        print("✅ MongoDB Connected Successfully!")
        return db
    except Exception as e:
        print("❌ MongoDB Connection Failed:", e)
        return None


In [ ]:
import os

# Create folders
os.makedirs("utils", exist_ok=True)

# Create __init__.py file (to mark it as a Python package)
with open("utils/__init__.py", "w") as f:
    f.write("")

# Create db.py file
db_code = """
import pymongo
from dotenv import load_dotenv
import os

load_dotenv()

MONGO_URI = os.getenv("MONGO_URI")

def get_db():
    try:
        client = pymongo.MongoClient(MONGO_URI)
        db = client["CodeGenieDB"]  # database name
        print("✅ MongoDB Connected Successfully!")
        return db
    except Exception as e:
        print("❌ MongoDB Connection Failed:", e)
        return None
"""

with open("utils/db.py", "w") as f:
    f.write(db_code)

print("✅ utils/db.py created successfully!")


In [ ]:
!pip install google-auth requests

In [ ]:
%%writefile app.py
# app.py
import streamlit as st
import pymongo
import bcrypt
import jwt
import datetime
import os
import random
import base64
from email.message import EmailMessage
import smtplib
from dotenv import load_dotenv
import pandas as pd
import plotly.express as px
import json
import requests
from streamlit_lottie import st_lottie
from PIL import Image # Import Image class from PIL
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, pipeline
import ast # Import ast for Python code parsing
import re # Import re for regex
# Google OAuth imports
import urllib.parse
import secrets
from google.oauth2 import id_token as google_id_token
from google.auth.transport import requests as google_auth_requests
import random
import time
import datetime
import smtplib
import ssl
from email.message import EmailMessage
import bcrypt  # pip install bcrypt

import warnings
warnings.filterwarnings("ignore")

# -----------------------------
# ⚙️ Setup & Configuration
# -----------------------------
load_dotenv()
MONGO_URI = os.getenv("MONGO_URI", "mongodb://localhost:27017/")
JWT_SECRET = os.getenv("JWT_SECRET", "supersecretkey")
SMTP_HOST = os.getenv("SMTP_HOST", "smtp.gmail.com")
SMTP_PORT = int(os.getenv("SMTP_PORT", 587))
SMTP_USER = os.getenv("SMTP_USER", "")
SMTP_PASS = os.getenv("SMTP_PASS", "")
EMAIL_FROM = os.getenv("EMAIL_FROM", "CodeGenie Support <no-reply@codegenie.ai>")
ADMIN_INITIAL_USER = os.getenv("ADMIN_INITIAL_USER", "admin@example.com") # Default if not in .env
ADMIN_INITIAL_PASS = os.getenv("ADMIN_INITIAL_PASS", "password") # Default if not in .env

GOOGLE_CLIENT_ID = os.getenv("GOOGLE_CLIENT_ID", "")
GOOGLE_CLIENT_SECRET = os.getenv("GOOGLE_CLIENT_SECRET", "")
# Example: "https://your-ngrok-id.ngrok.io" or "http://localhost:8501"
APP_BASE_URL = os.getenv("APP_BASE_URL")


client = pymongo.MongoClient(MONGO_URI)
db = client["codegenie"]
users_collection = db["users"]
history_collection = db["history"]
reviews_collection = db["reviews"]
files_collection = db["files"] # Collection for files
logs_collection = db["logs"] # Collection for logs
requests_collection = db["requests"] # Collection for admin requests

#--------------------------------
#--------------------------------


def make_google_oauth_url(state):
    auth_endpoint = "https://accounts.google.com/o/oauth2/v2/auth"
    scope = "openid email profile"
    redirect_uri = APP_BASE_URL.rstrip("/") + "/"
    params = {
        "response_type": "code",
        "client_id": GOOGLE_CLIENT_ID,
        "redirect_uri": redirect_uri,
        "scope": scope,
        "state": state,
        "prompt": "select_account",
        "access_type": "offline",
    }
    return auth_endpoint + "?" + urllib.parse.urlencode(params)

def exchange_code_for_tokens(code):
    token_url = "https://oauth2.googleapis.com/token"
    redirect_uri = APP_BASE_URL.rstrip("/") + "/"
    data = {
        "code": code,
        "client_id": GOOGLE_CLIENT_ID,
        "client_secret": GOOGLE_CLIENT_SECRET,
        "redirect_uri": redirect_uri,
        "grant_type": "authorization_code",
    }
    resp = requests.post(token_url, data=data, timeout=10)
    resp.raise_for_status()
    return resp.json()  # contains access_token, id_token, refresh_token (optional)

def verify_google_id_token(id_token):
    # verifies signature and audience (client id)
    try:
        claims = google_id_token.verify_oauth2_token(id_token, google_auth_requests.Request(), GOOGLE_CLIENT_ID)
        # typical claims: email, email_verified, name, picture, sub (google user id)
        return claims
    except Exception as e:
        return None

def login_via_google(claims):
    """
    Upsert user into users_collection with auth_method 'google' and set session.
    Returns username/email used for local session.
    """
    email = claims.get("email")
    if not email:
        raise ValueError("No email in Google claims")
    # Upsert user document (no password). Keep role default 'user' unless already set.
    existing = users_collection.find_one({"username": email})
    if existing:
        users_collection.update_one({"username": email}, {"$set": {"auth_method": "google", "google_sub": claims.get("sub"), "name": claims.get("name")}})
        role = existing.get("role", "user")
    else:
        users_collection.insert_one({
            "username": email,
            "role": "user",
            "created_at": datetime.datetime.utcnow(),
            "auth_method": "google",
            "google_sub": claims.get("sub"),
            "name": claims.get("name"),
        })
        role = "user"
    # create session JWT and set session state
    token = generate_token(email)
    st.session_state.jwt_token = token
    st.session_state.username = email
    st.session_state.role = role
    log_action("login_google", username=email, meta={"sub": claims.get("sub")})
    return email


# -----------------------------
# 💡 title Helper
# -----------------------------
def render_title_header():
    st.markdown("""
          <h2 style='text-align:center; font-weight:800; color:#00FFFF;'>
              🤖 CodeGenie - The Code Generator & Explainer
          </h2>
          <p style='text-align:center; color:#b8b8b8;'>Let the genie do it for you ;)</p>
          <hr style="border: 1px solid rgba(255,255,255,0.1);"/>
      """, unsafe_allow_html=True)
    # st.markdown(
    #     """
    #     <div style="
    #         text-align:center;
    #         font-size: 42px;
    #         font-weight: 800;
    #         padding: 14px 0;
    #         border: 2px solid rgba(0, 255, 255, 0.5);
    #         border-radius: 16px;
    #         margin-bottom: 26px;
    #         background: rgba(0, 0, 0, 0.25);
    #         backdrop-filter: blur(6px);
    #         color: white;
    #         letter-spacing: 1.5px;
    #         box-shadow: 0 0 5px rgba(0, 255, 255, 0.35);
    #     ">
    #         <span style="color:#00ffff;">⚡ Code</span><span style="color:#b366ff;">Genie</span>
    #     </div>
    #     """,
    #     unsafe_allow_html=True
    # )

# -----------------------------
# 💡 review Helper
# -----------------------------
def save_user_review(username, mode, query, output, model, language, rating, review_text=None, response_time_ms=None):
    """
    Saves the review document to reviews_collection.
    Required fields: username, mode, query, output, model, language, rating, timestamp.
    Optional: review_text, response_time_ms
    """
    try:
        doc = {
            "username": username,
            "mode": mode,
            "query": query,
            "output": output,
            "model": model,
            "language": language,
            "rating": int(rating),
            "review": review_text if review_text else "",
            "response_time_ms": response_time_ms if response_time_ms is not None else None,
            "timestamp": datetime.datetime.utcnow()
        }
        reviews_collection.insert_one(doc)
        log_action("submit_review_automated", username=username, meta={"mode": mode, "model": model, "rating": rating})
        return True, "Saved"
    except Exception as e:
        log_action("submit_review_failed", username=username, status="failed", meta={"error": str(e)})
        return False, str(e)



# -----------------------------
# 💡 Logging Helper
# -----------------------------
def log_action(action, username="system", status="success", meta=None):
    """Logs an action to the logs collection."""
    log_entry = {
        "timestamp": datetime.datetime.utcnow(),
        "action": action,
        "username": username,
        "status": status,
        "meta": meta if meta is not None else {}
    }
    try:
        logs_collection.insert_one(log_entry)
    except Exception as e:
        print(f"Error logging action {action}: {e}")


# -----------------------------
# 🚠️ Utils
# -----------------------------
def df_from_cursor(cursor):
    """Converts a MongoDB cursor to a pandas DataFrame."""
    try:
        df = pd.DataFrame(list(cursor))
        # Convert ObjectId to string for compatibility
        if '_id' in df.columns:
            df['_id'] = df['_id'].astype(str)
        return df
    except Exception as e:
        print(f"Error converting cursor to DataFrame: {e}")
        return pd.DataFrame()

# -----------------------------
# 🎨 UI Theme & Styling
# -----------------------------
def apply_neon_css():
    st.markdown("""
    <style>
    .stApp {
        background-color: #0a0a0a;
        background-image: radial-gradient(circle at top left, #0f0f3d, #000000);
        color: #eaeaea;
        font-family: 'JetBrains Mono', monospace;
    }
    .neon-card {
        background: rgba(255,255,255,0.03);
        border: 1px solid rgba(255,255,255,0.1);
        border-radius: 12px;
        box-shadow: 0 0 20px rgba(0,255,255,0.1);
        padding: 20px;
    }
    .stButton>button {
        background: linear-gradient(90deg,#00FFFF,#8A2BE2);
        color:#000;
        font-weight:700;
        border:none;
        border-radius:6px;
        transition:0.3s;
    }
    .stButton>button:hover {
        transform:scale(1.05);
        background:linear-gradient(90deg,#FF00FF,#8A2BE2);
        color:#fff;
        box-shadow:0 0 25px #FF00FF;
    }
    h1,h2,h3 {
        background:linear-gradient(90deg,#00FFFF,#FF00FF);
        -webkit-background-clip:text;
        -webkit-text-fill-color:transparent;
    }
    </style>
    """, unsafe_allow_html=True)

# -----------------------------
# 🔐 Authentication Helpers
# -----------------------------
def create_user(username, password, role="user"):
    if users_collection.find_one({"username": username}):
        return False, "User already exists."
    hashed = bcrypt.hashpw(password.encode(), bcrypt.gensalt())
    users_collection.insert_one(
        {
            "username": username,
            "password": hashed,
            "role": role,
            "created_at": datetime.datetime.utcnow(),
            "auth_method": "email",
        }
    )
    log_action("create_user", username=username, meta={"role": role})
    return True, "Account created successfully!"

def verify_user(username, password):
    user = users_collection.find_one({"username": username})
    if user and bcrypt.checkpw(password.encode(), user["password"]):
        log_action("login", username=username, status="success")
        return True
    log_action("login", username=username, status="failed")
    return False

def generate_token(username):
    user = users_collection.find_one({"username": username})
    payload = {"username": username, "role": user.get("role", "user"),
               "exp": datetime.datetime.utcnow() + datetime.timedelta(hours=8)}
    return jwt.encode(payload, JWT_SECRET, algorithm="HS256")

def decode_token(token):
    try:
        return jwt.decode(token, JWT_SECRET, algorithms=["HS256"])
    except Exception:
        return None

# -----------------------------
# 🔑 OTP Password Reset
# -----------------------------
def generate_otp():
    return f"{random.randint(100000,999999)}"

# def send_otp_email(to_email, otp):
#     try:
#         msg = EmailMessage()
#         msg["Subject"] = "CodeGenie OTP Verification"
#         msg["From"] = EMAIL_FROM
#         msg["To"] = to_email
#         msg.set_content(f"Your OTP code for password reset is: {otp}\nIt will expire in 5 minutes.")
#         server = smtplib.SMTP(SMTP_HOST, SMTP_PORT)
#         server.starttls()
#         server.login(SMTP_USER, SMTP_PASS)
#         server.send_message(msg)
#         server.quit()
#         log_action("send_otp_email", username=to_email, status="success")
#         return True
#     except Exception as e:
#         log_action("send_otp_email", username=to_email, status="failed", meta={"error": str(e)})
#         st.warning(f"SMTP Error: {e}")
#         return False

def send_otp_email_with_link(to_email, otp, reset_jwt, app_base_url=None):
    """
    Sends OTP and includes a reset link containing reset_jwt as a query param.
    app_base_url: e.g. "https://your-app-url" (no trailing slash). If None, link will include token only.
    """
    try:
        msg = EmailMessage()
        msg["Subject"] = "CodeGenie OTP Verification & Reset Link"
        msg["From"] = EMAIL_FROM
        msg["To"] = to_email

        link_text = ""
        if app_base_url:
            # Make a url such that clicking opens your Streamlit app with ?reset_jwt=<token>
            # Make sure your app is reachable at this domain. If testing locally, use ngrok URL.
            link = f"{app_base_url}?reset_jwt={reset_jwt}"
            link_text = f"\nOr click this link to open the reset page:\n{link}\n"
        else:
            link_text = f"\nReset token (paste into app if needed): {reset_jwt}\n"

        body = f"""
Hello,

Your CodeGenie password reset OTP is: {otp}
This OTP is valid for 5 minutes.

{link_text}

If you did not request this, ignore this email.

— CodeGenie
"""
        msg.set_content(body)

        server = smtplib.SMTP(SMTP_HOST, SMTP_PORT)
        server.starttls()
        server.login(SMTP_USER, SMTP_PASS)
        server.send_message(msg)
        server.quit()
        log_action("send_otp_email", username=to_email, status="success")
        return True
    except Exception as e:
        log_action("send_otp_email", username=to_email, status="failed", meta={"error": str(e)})
        st.warning(f"SMTP Error: {e}")
        return False


dark_theme_css = """
<style>
    .main { background-color: #0E1117; color: #FAFAFA; }
    .auth-container { background-color: #161B22; border: 1px solid #30363D; }
    h1, h2, h3, h4, h5, h6 { color: #C9D1D9; }
</style>
"""

# -----------------------------
# 🌈 Load Lottie Animations
# -----------------------------
def load_lottie(url: str):
    r = requests.get(url)
    if r.status_code != 200:
        return None
    return r.json()

lottie_ai = load_lottie("https://assets2.lottiefiles.com/packages/lf20_tfb3estd.json")
lottie_review = load_lottie("https://assets10.lottiefiles.com/packages/lf20_wd1udlcz.json")
lottie_login = load_lottie("https://assets2.lottiefiles.com/packages/lf20_zrqthn6o.json")

# -----------------------------
# Models (lazy load)
# -----------------------------
MODELS_TO_LOAD = {
    "deepseek-ai/deepseek-coder-1.3b-instruct": "DeepSeek-Coder-1.3B",
    "microsoft/phi-2": "Phi-2-2.7B",
    "facebook/codebart-base": "codebart-base"
}

@st.cache_resource
def load_all_models():
    loaded = {}
    st.info(f"Initiating model loading for {len(MODELS_TO_LOAD)} model(s)... This might take a while the first time.")
    for model_id, name in MODELS_TO_LOAD.items():
        st.write(f"Loading {name} ({model_id})...")
        try:
            dtype = torch.bfloat16 if torch.cuda.is_available() and torch.cuda.is_bf16_supported() else torch.float16
            model = AutoModelForCausalLM.from_pretrained(model_id, torch_dtype=dtype, device_map="auto", trust_remote_code=True)
            tokenizer = AutoTokenizer.from_pretrained(model_id, trust_remote_code=True)
            loaded[model_id] = (model, tokenizer)
            st.success(f"✅ Loaded {name} successfully.")
        except Exception as e:
            st.error(f"🚨 Error loading {name} ({model_id}): {e}")
            loaded[model_id] = (None, None) # Store None to indicate failure
    st.info("Model loading complete.")
    return loaded


# -----------------------------
# Code Functions
# -----------------------------
def explain_code(prompt, language="python"):
    """Explain code using AST for Python, and a fine-tuned CodeBERT for others."""
    if language.lower() == "python":
        try:
            tree = ast.parse(prompt)
            explanation = "✅ **Python Code Structure (AST):**\n```\n" + ast.dump(tree, indent=4) + "\n```"
            return explanation
        except Exception as e:
            return f"⚠️ Error parsing Python code: {e}\n\nCould not provide AST explanation."
    else:
        # Placeholder for explaining non-Python code with a model (CodeBERT or similar)
        return f"💡 **AI-Generated Explanation (Placeholder):**\n\nExplanation for {language} code: \n\nThis code performs XYZ operations."

def generate_code(prompt, language, model_selectbox_name, loaded_models_dict):
    """Generates code using a pre-loaded model from the provided dictionary."""
    model_name_to_hf_id = {
        "DeepSeek-Coder-1.3B": "deepseek-ai/deepseek-coder-1.3b-instruct",
        "Phi-2-2.7B": "microsoft/phi-2",
        "codebart-base": "facebook/codebart-base"
    }
    hf_model_id = model_name_to_hf_id.get(model_selectbox_name)

    if not hf_model_id:
        return f"Error: Model name '{model_selectbox_name}' not mapped correctly."

    if hf_model_id not in loaded_models_dict or loaded_models_dict[hf_model_id][0] is None:
        return f"Error: Model '{model_selectbox_name}' failed to load during startup."

    model, tokenizer = loaded_models_dict[hf_model_id]

    try:
        # Use a pipeline with the pre-loaded model and tokenizer
        pipe = pipeline("text-generation", model=model, tokenizer=tokenizer, max_new_tokens=512)

        # Specific prompt formatting (adjust as needed for different models)
        # This is a basic example, prompt engineering is crucial for better results
        formatted_prompt = f"Write {language} code that does the following:\n{prompt}\n\n```{language.lower()}\n"

        st.write(f"Generating code using {model_selectbox_name} for {language}...")
        result = pipe(formatted_prompt, max_new_tokens=512) # Added max_new_tokens to pipeline call
        generated_text = result[0]['generated_text']

        # Basic post-processing to extract potentially just the code block
        # Look for the first code block after the prompt
        # Adjusted regex to be less specific about language in the ``` block
        code_block_match = re.search(r"```(?:[a-zA-Z0-9_-]*)\n(.*?)```", generated_text, re.DOTALL)
        if code_block_match:
            extracted_code = code_block_match.group(1).strip()
            return f"💡 **AI-Generated Code:**\n\n```{language.lower()}\n{extracted_code}\n```"
        else:
             # If no clear code block is found, return the text after the initial prompt marker
            response_marker = f"```" # Assuming the prompt ends with ```
            start_index = generated_text.find(response_marker)
            if start_index != -1:
                # Return content after the marker
                content_after_marker = generated_text[start_index + len(response_marker):].strip()
                # If the content looks like code, wrap it
                if content_after_marker.startswith(('def ', 'class ', 'import ', 'SELECT ', '<!DOCTYPE', '{', '[')): # Simple heuristic
                     return f"💡 **AI-Generated Text (Potential Code):**\n\n```{language.lower()}\n{content_after_marker}\n```"
                else:
                     return f"💡 **AI-Generated Text:**\n\n{content_after_marker}"

            else:
                 # Fallback: return the full generated text if no marker is found
                return f"💡 **AI-Generated Text (Full Output):**\n\n{generated_text.strip()}"


    except Exception as e:
        st.error(f"Error during generation with {model_selectbox_name}: {e}")
        return f"Error during generation: {e}"


# -----------------------------
# 🧠 Main Workspace
# -----------------------------
def workspace_page():
    st.title("💻 CodeGenie Workspace")

    # Load models at the beginning of the workspace page function
    # This ensures models are loaded/available when needed
    with st.expander("click to see the info regarding loading models....."):
      loaded_models = load_all_models()
    available_model_names = [name for id, name in MODELS_TO_LOAD.items() if loaded_models.get(id) and loaded_models[id][0] is not None]


    mode = st.radio("Select Mode", ["Explain Code", "Generate Code"], horizontal=True)
    lang = st.selectbox("Language", ["Python","C++","JavaScript","SQL", "Java", "HTML", "CSS", "Other"]) # Added more languages
    prompt = st.text_area("Enter your prompt or code snippet:", height=200)

    model_selectbox_name = None
    if mode == "Generate Code":
        if not available_model_names:
            st.warning("⚠️ No code generation models loaded successfully. Generation is unavailable.")
        else:
             # Only show selectbox if models are available
            model_selectbox_name = st.selectbox("Select Generation Model", available_model_names)


    if st.button("Run AI"):
        if prompt.strip() == "":
            st.warning("Please enter a prompt.")
            return

        result = ""
        model_used = "N/A" # Default
        if mode == "Explain Code":
            model_used = "AST Parser (Python) / Placeholder AI Explainer" # Update model name description
            start = time.perf_counter()
            try:
                  # your existing call
                result = explain_code(prompt, lang) # Pass language to explain_code
                succeeded = True
            except Exception as e:
                # if model call fails, still capture elapsed and log error info
                result = None
                succeeded = False
                err_str = str(e)
            finally:
                end = time.perf_counter()
                elapsed_ms = int((end - start) * 1000)  # milliseconds as integer


        elif mode == "Generate Code":
             if not available_model_names or not model_selectbox_name:
                 st.warning("Cannot generate code: No models available or selected.")
                 return # Stop execution if no model is available/selected

             model_used = model_selectbox_name
             # Pass the dictionary of loaded models AND language here

             start = time.perf_counter()
             try:
                   # your existing call
                 result = generate_code(prompt, lang, model_selectbox_name, loaded_models)
                 succeeded = True
             except Exception as e:
                # if model call fails, still capture elapsed and log error info
                 result = None
                 succeeded = False
                 err_str = str(e)
             finally:
                 end = time.perf_counter()
                 elapsed_ms = int((end - start) * 1000)  # milliseconds as integer





        st.subheader("Result:")
        st.markdown(result) # Display the generated/explained code

        # ---------- Ask for review (required rating + optional text) ----------
        # Only show review prompt for logged-in users
        if st.session_state.get("username"):
            # Use a session_state flag to show prompt only once per result
            # Build a unique key so repeated runs don't keep re-showing the same review prompt
            review_flag_key = f"asked_review_{hash((prompt, result, model_used, lang))}"
            if not st.session_state.get(review_flag_key, False):
                with st.expander("⭐ Help us improve — leave a rating and optional review", expanded=True):
                    # Use a simple form so submission is atomic
                    with st.form(key=f"review_form_{review_flag_key}"):
                        rating = st.slider("Rating (required)", min_value=1, max_value=5, value=5, help="Rate how helpful the output was (1-5)")
                        review_text = st.text_area("Optional: Write a short review (what worked / what didn't)", height=120)
                        submitted = st.form_submit_button("Submit Review")
                        if submitted:
                            # Validate required rating
                            if rating is None:
                                st.error("Please provide a rating (1-5).")
                            else:
                                # Save review to DB
                                ok, msg = save_user_review(
                                    username=st.session_state.username,
                                    mode=mode,
                                    query=prompt,
                                    output=result,
                                    model=model_used,
                                    language=lang,
                                    rating=rating,
                                    review_text=review_text,
                                    response_time_ms=None # If you capture response time, pass it here
                                )
                                if ok:
                                    st.success("Thanks — your rating was saved!")
                                    st.session_state[review_flag_key] = True
                                else:
                                    st.error(f"Could not save review: {msg}")
            else:
                st.info("Thanks — you already reviewed this result.")
        else:
            st.info("Log in to submit ratings and reviews. (Only logged-in users' feedback is saved.)")


        # Save to history (only if a meaningful result was obtained)
        if result and not result.startswith("Error:") and not result.startswith("Warning:"):
             try:
                history_collection.insert_one({
                    "username": st.session_state.username,
                    "mode": mode,
                    "language": lang,
                    "prompt": prompt,
                    "result": result,
                    "model": model_used, # Store the model used
                    "response_time_ms": elapsed_ms,
                    "error":err_str if not succeeded else None,
                    "timestamp": datetime.datetime.utcnow()
                })
                st.success("✅ Result added to your history!")
             except Exception as e:
                st.warning(f"Could not save to history: {e}")
        elif result.startswith("Error:"):
            st.error("An error occurred during processing. Not saved to history.")


# -----------------------------
# ⟲ History Page
# -----------------------------
def history_page():
    st.title("🕓 Your Activity History")

    # --- Ensure session_state defaults (safe to call anytime) ---
    if "history_page_index" not in st.session_state:
        st.session_state.history_page_index = 0
    if "history_page_size" not in st.session_state:
        st.session_state.history_page_size = 25
    # Keys used by widgets:
    if "history_search_text" not in st.session_state:
        st.session_state.history_search_text = ""
    if "history_model_filter" not in st.session_state:
        st.session_state.history_model_filter = []
    if "history_lang_filter" not in st.session_state:
        st.session_state.history_lang_filter = []

    st.markdown("Use the search box to find past prompts/outputs. Filter by model, language and date range.")

    # --- Top-row controls ---
    col1, col2, col3 = st.columns([3,2,2])
    with col1:
        # Bound to st.session_state["history_search_text"]
        search_text = st.text_input(
            "Search (prompt / output / model / language)",
            key="history_search_text",
            placeholder="e.g. pagination bug, python, DeepSeek..."
        )
    with col2:
        start_date = st.date_input(
            "From",
            value=(datetime.datetime.utcnow() - datetime.timedelta(days=30)).date(),
            key="history_start_date"
        )
    with col3:
        end_date = st.date_input(
            "To",
            value=datetime.datetime.utcnow().date(),
            key="history_end_date"
        )

    # Build base query (only show user's history)
    query = {"username": st.session_state.username}

    # Date range
    start_dt = datetime.datetime.combine(start_date, datetime.time.min)
    end_dt = datetime.datetime.combine(end_date, datetime.time.max)
    query["timestamp"] = {"$gte": start_dt, "$lte": end_dt}

    # Add text search across multiple fields if provided
    if search_text and search_text.strip():
        pattern = {"$regex": search_text.strip(), "$options": "i"}
        query["$or"] = [
            {"prompt": pattern},
            {"result": pattern},
            {"model": pattern},
            {"language": pattern}
        ]

    # Preview DB to populate model & language filter options (scoped to user + date range)
    try:
        preview_cursor = history_collection.find(
            {"username": st.session_state.username, "timestamp": {"$gte": start_dt, "$lte": end_dt}},
            {"model":1, "language":1}
        ).limit(1000)
        preview_df = pd.DataFrame(list(preview_cursor))
        available_models = sorted([m for m in preview_df['model'].dropna().unique()]) if (not preview_df.empty and 'model' in preview_df.columns) else []
        available_languages = sorted([l for l in preview_df['language'].dropna().unique()]) if (not preview_df.empty and 'language' in preview_df.columns) else []
    except Exception:
        available_models = []
        available_languages = []

    # --- Single horizontal row for model filter, language filter, and page size ---
    colf1, colf2, colf3 = st.columns([3,3,1])
    with colf1:
        model_filter = st.multiselect("Model (filter)", options=available_models, key="history_model_filter")
    with colf2:
        lang_filter = st.multiselect("Language (filter)", options=available_languages, key="history_lang_filter")
    with colf3:
        page_size = st.selectbox(
            "Page size",
            options=[10, 25, 50, 100],
            index=[10,25,50,100].index(st.session_state.get("history_page_size", 25)),
            key="history_page_size"
        )

    # Apply model/language filters to the query
    if model_filter:
        query["model"] = {"$in": model_filter}
    if lang_filter:
        query["language"] = {"$in": lang_filter}

    # Reconcile session page index and current values
    current_page = st.session_state.get("history_page_index", 0)

    # Count total matches (safe)
    try:
        total_count = history_collection.count_documents(query)
    except Exception as e:
        st.error(f"Error counting history: {e}")
        return

    total_pages = max(1, (total_count + page_size - 1) // page_size)
    if current_page < 0:
        current_page = 0
        st.session_state.history_page_index = 0
    if current_page >= total_pages:
        current_page = total_pages - 1
        st.session_state.history_page_index = current_page

    # Prev / Next buttons and info
    col_prev, col_next, col_info = st.columns([1,1,4])
    with col_prev:
        if st.button("◀ Prev", key="history_prev") and st.session_state.history_page_index > 0:
            st.session_state.history_page_index -= 1
            st.rerun()
    with col_next:
        if st.button("Next ▶", key="history_next") and st.session_state.history_page_index < total_pages - 1:
            st.session_state.history_page_index += 1
            st.rerun()
    with col_info:
        st.markdown(f"**Showing page {current_page+1}/{total_pages}** — {total_count} result(s) matched")

    # Fetch the page of results
    skip = current_page * page_size
    try:
        cursor = history_collection.find(query).sort("timestamp", -1).skip(skip).limit(page_size)
        records = list(cursor)
    except Exception as e:
        st.error(f"Error fetching history: {e}")
        return

    if not records:
        st.info("No history found for the given criteria.")
        return

    # Convert to DataFrame for display
    df = pd.DataFrame(records)
    if 'timestamp' in df.columns:
        df['timestamp'] = pd.to_datetime(df['timestamp']).dt.strftime("%Y-%m-%d %H:%M:%S")
    display_df = df.copy()
    display_df = display_df.head(page_size)
    if '_id' in display_df.columns:
        display_df['_id'] = display_df['_id'].astype(str)

    preferred_cols = ['timestamp','mode','language','model','prompt','result']
    display_cols = [c for c in preferred_cols if c in display_df.columns] + [c for c in display_df.columns if c not in preferred_cols]
    st.dataframe(display_df[display_cols], use_container_width=True, hide_index=True)

    # Export filtered results as CSV
    csv = display_df.to_csv(index=False).encode()
    st.download_button("⬇️ Export Filtered History (CSV)", csv, file_name=f"{st.session_state.username}_history_filtered.csv")

    # Record inspector
    st.markdown("---")
    st.subheader("Inspect a record")
    try:
        ids = display_df['_id'].tolist() if '_id' in display_df.columns else []
        id_to_show = st.selectbox("Select record to inspect (by id)", options=ids, key="history_inspect_select")
        if id_to_show:
            rec = next((r for r in records if str(r.get('_id')) == id_to_show), None)
            if rec:
                ts = rec.get('timestamp')
                ts_str = ts if isinstance(ts, str) else (ts.strftime("%Y-%m-%d %H:%M:%S") if isinstance(ts, datetime.datetime) else str(ts))
                st.markdown(f"**Timestamp:** {ts_str}")
                st.markdown(f"**Mode:** {rec.get('mode')} — **Model:** {rec.get('model')} — **Language:** {rec.get('language')}")
                st.markdown("**Prompt:**")
                st.code(rec.get('prompt',''), language='text')
                st.markdown("**Result:**")
                st.code(rec.get('result','')[:10000], language='text')  # truncate for display
    except Exception:
        pass


# # -----------------------------
# # ⭐️ Review Page
# # -----------------------------
# def review_page():
#     st.title("⭐️ Share Your Feedback")
#     st_lottie(lottie_review, height=120, key="revAnim")
#     rating = st.slider("Rate CodeGenie (1-5)", 1,5,5)
#     feedback = st.text_area("Your Feedback:")
#     if st.button("Submit Review"):
#         if feedback.strip():
#             reviews_collection.insert_one({
#                 "username": st.session_state.username,
#                 "rating": rating,
#                 "review": feedback.strip(),
#                 "timestamp": datetime.datetime.utcnow()
#             })
#             log_action("submit_review", username=st.session_state.username, meta={"rating": rating})
#             st.success("Thank you for your feedback! ⭐️")
#         else:
#             st.warning("Please enter some feedback.")


import os, json, time, datetime
import streamlit as st
from transformers import pipeline
# (Assume you have a load_all_models() function and apply_neon_css() already defined)

def general_chat_page():
    apply_neon_css()

    st.markdown("""
        <h2 style='text-align:center; font-weight:800; color:#00FFFF;'>
            🤖 CodeGenie AI Chat
        </h2>
        <p style='text-align:center; color:#b8b8b8;'>Ask anything — the AI remembers and responds like ChatGPT.</p>
        <hr style="border: 1px solid rgba(255,255,255,0.1);"/>
    """, unsafe_allow_html=True)

    CHAT_DIR = "chat_history"
    os.makedirs(CHAT_DIR, exist_ok=True)

    # ---------------------------
    # 🧠 Initialize session memory
    # ---------------------------
    if "chat_history" not in st.session_state:
        st.session_state.chat_history = []
    if "current_chat" not in st.session_state:
        st.session_state.current_chat = None
    if "selected_chat" not in st.session_state:
        st.session_state.selected_chat = "🆕 New Chat"
    if "rename_input" not in st.session_state:
        st.session_state.rename_input = ""

    # ---------------------------
    # 💾 Helper functions
    # ---------------------------
    def list_chats():
        files = [f[:-5] for f in os.listdir(CHAT_DIR) if f.endswith(".json")]
        return sorted(files, key=lambda x: x.lower())

    def save_chat(title, history):
        """Save only last 10 messages"""
        with open(f"{CHAT_DIR}/{title}.json", "w", encoding="utf-8") as f:
            json.dump(history[-10:], f, indent=2, ensure_ascii=False)

    def load_chat(title):
        """Load chat messages"""
        path = f"{CHAT_DIR}/{title}.json"
        if os.path.exists(path):
            with open(path, "r", encoding="utf-8") as f:
                return json.load(f)
        return []

    def append_message(role, content):
        """Add message to session"""
        st.session_state.chat_history.append({"role": role, "content": content})
        st.session_state.chat_history = st.session_state.chat_history[-10:]

    def rename_chat(old_name, new_name):
        """Rename chat file"""
        new_name = new_name.strip()
        if not new_name or new_name == old_name:
            return False
        old_path = os.path.join(CHAT_DIR, f"{old_name}.json")
        new_path = os.path.join(CHAT_DIR, f"{new_name}.json")
        if os.path.exists(new_path):
            st.warning("⚠️ A chat with that name already exists.")
            return False
        os.rename(old_path, new_path)
        st.session_state.current_chat = new_name
        st.session_state.selected_chat = new_name
        return True

    # ---------------------------
    # 🧭 Sidebar Chat List
    # ---------------------------
    st.sidebar.header("💬 Conversations")

    chats = list_chats()
    selected = st.sidebar.selectbox("Select Chat", ["🆕 New Chat"] + chats,
                                    index=(["🆕 New Chat"] + chats).index(st.session_state.selected_chat)
                                    if st.session_state.selected_chat in ["🆕 New Chat"] + chats else 0)

    # New Chat Button
    if st.sidebar.button("🆕 New Chat"):
        if st.session_state.current_chat:
            save_chat(st.session_state.current_chat, st.session_state.chat_history)
        st.session_state.chat_history = []
        st.session_state.current_chat = None
        st.session_state.selected_chat = "🆕 New Chat"
        st.rerun()

    # If a different chat is selected
    if selected != st.session_state.selected_chat:
        if st.session_state.current_chat:
            save_chat(st.session_state.current_chat, st.session_state.chat_history)
        st.session_state.selected_chat = selected
        if selected == "🆕 New Chat":
            st.session_state.chat_history = []
            st.session_state.current_chat = None
        else:
            st.session_state.chat_history = load_chat(selected)
            st.session_state.current_chat = selected

    # ---------------------------
    # ✏️ Rename Chat Feature
    # ---------------------------
    if st.session_state.current_chat:
        st.sidebar.markdown("### ✏️ Rename this chat")
        st.session_state.rename_input = st.sidebar.text_input("New Chat Name:", st.session_state.current_chat)
        if st.sidebar.button("Save New Name"):
            if rename_chat(st.session_state.current_chat, st.session_state.rename_input):
                st.sidebar.success("✅ Chat renamed successfully!")
                st.rerun()
            else:
                st.sidebar.error("⚠️ Invalid or duplicate name.")

        # Delete chat
        if st.sidebar.button("🗑️ Delete Chat"):
            path = f"{CHAT_DIR}/{st.session_state.current_chat}.json"
            if os.path.exists(path):
                os.remove(path)
            st.session_state.chat_history = []
            st.session_state.current_chat = None
            st.session_state.selected_chat = "🆕 New Chat"
            st.sidebar.success("🗑️ Chat deleted successfully.")
            st.rerun()

    # ---------------------------
    # 💬 Chat Display UI
    # ---------------------------
    st.markdown("""
        <style>
        .chat-container {background-color: rgba(255,255,255,0.02);border-radius:15px;padding:15px;max-height:60vh;overflow-y:auto;}
        .user-bubble {text-align:right;background:linear-gradient(90deg,rgba(0,255,255,0.2),rgba(0,255,255,0.05));border-right:4px solid #00FFFF;border-radius:12px;padding:10px;margin:8px 0;color:#E0FFFF;font-size:15px;}
        .ai-bubble {text-align:left;background:linear-gradient(90deg,rgba(180,100,255,0.15),rgba(100,50,255,0.05));border-left:4px solid #B366FF;border-radius:12px;padding:10px;margin:8px 0;color:#D9C9FF;font-size:15px;}
        </style>
    """, unsafe_allow_html=True)

    st.markdown('<div class="chat-container">', unsafe_allow_html=True)
    for msg in st.session_state.chat_history:
        role_class = "user-bubble" if msg["role"] == "user" else "ai-bubble"
        prefix = "👤 You:" if msg["role"] == "user" else "🤖 AI:"
        st.markdown(f"<div class='{role_class}'><b>{prefix}</b> {msg['content']}</div>", unsafe_allow_html=True)
    st.markdown("</div>", unsafe_allow_html=True)

    # ---------------------------
    # 💭 Input box
    # ---------------------------
    user_input = st.text_area("💬 Type your message:", placeholder="Ask something...", height=100)
    send = st.button("Send", use_container_width=True)

    # ---------------------------
    # 🚀 Chat logic
    # ---------------------------
    if send and user_input.strip():
        user_input = user_input.strip()
        append_message("user", user_input)

        # Auto title for new chat (no datetime)
        if not st.session_state.current_chat:
            clean_title = "_".join(user_input.split()[:3])
            st.session_state.current_chat = clean_title
            st.session_state.selected_chat = clean_title

        with st.spinner("🤖 AI is thinking..."):
            loaded_models = load_all_models()
            hf_model_id = "microsoft/phi-2"
            model, tokenizer = loaded_models[hf_model_id]
            pipe = pipeline("text-generation", model=model, tokenizer=tokenizer, max_new_tokens=300)

            # Construct conversation context
            context = "\n".join(
                [f"User: {m['content']}" if m["role"] == "user" else f"Assistant: {m['content']}"
                 for m in st.session_state.chat_history[-10:]]
            )
            prompt = f"{context}\nUser: {user_input}\nAssistant:"

            result = pipe(prompt, max_new_tokens=200, do_sample=True, temperature=0.7)
            response = result[0]["generated_text"].split("Assistant:")[-1].strip()

            append_message("assistant", response)
            save_chat(st.session_state.current_chat, st.session_state.chat_history)
            st.rerun()

    # ---------------------------
    # ↩️ Enter key to send
    # ---------------------------
    st.markdown("""
        <script>
        const textarea = window.parent.document.querySelector('textarea');
        if (textarea) {
            textarea.addEventListener('keydown', function(e) {
                if (e.key === 'Enter' && !e.shiftKey) {
                    e.preventDefault();
                    const sendBtn = window.parent.document.querySelector('button[kind=primary]');
                    if (sendBtn) sendBtn.click();
                }
            });
        }
        </script>
    """, unsafe_allow_html=True)



# -----------------------------
# 👤 Profile Page
# -----------------------------
def profile_page():
    st.title("👤 Your Profile")
    user = users_collection.find_one({"username": st.session_state.username})

    # Profile card (keeps your neon styling)
    if user:
        st.markdown(f"""
        <div class='neon-card'>
        <b>Username:</b> {user.get("display_name", "NA")}<br>
        <b>Email:</b> {user['username']}<br>
        <b>Role:</b> {user.get('role','user')}<br>
        <b>Joined:</b> {user['created_at'].strftime('%Y-%m-%d') if user.get('created_at') else 'N/A'}<br>
        </div>
        """, unsafe_allow_html=True)
    else:
        st.error("User not found in database.")
        return

    st.markdown("---")

    # -------------------------
    # Change password section
    # -------------------------
    st.subheader("🔁 Change Password")

    # Use a form so the operation is atomic and UX is tidy
    with st.form("change_password_form"):
        cur_pw = st.text_input("Current password", type="password", key="chg_cur_pw")
        new_pw = st.text_input("New password", type="password", key="chg_new_pw")
        new_pw_conf = st.text_input("Confirm new password", type="password", key="chg_new_pw_conf")
        submit_change = st.form_submit_button("Update Password")

    if submit_change:
        # Basic validation
        if not cur_pw or not new_pw or not new_pw_conf:
            st.error("Please fill all fields.")
        elif new_pw != new_pw_conf:
            st.error("New passwords do not match.")
        else:
            # Optional: basic password strength checks
            pw_errors = []
            if len(new_pw) < 8:
                pw_errors.append("Password should be at least 8 characters.")
            if not any(c.isdigit() for c in new_pw):
                pw_errors.append("Password should contain at least one digit.")
            if not any(c.isalpha() for c in new_pw):
                pw_errors.append("Password should contain at least one letter.")
            # You can add more checks (special chars, uppercase, etc.)

            if pw_errors:
                for e in pw_errors:
                    st.warning(e)
                st.error("Please fix the password strength issues above.")
            else:
                # Verify current password
                try:
                    db_user = users_collection.find_one({"username": st.session_state.username})
                    if not db_user:
                        st.error("User not found. Please re-login.")
                        return

                    # If user has no password field or uses alternate auth, disallow
                    if "password" not in db_user:
                        st.error("Password change is not available for this account (no password set).")
                        return

                    if not bcrypt.checkpw(cur_pw.encode(), db_user["password"]):
                        st.error("Current password is incorrect.")
                        log_action("change_password_failed_wrong_current", username=st.session_state.username, status="failed")
                    else:
                        # All good; update the password and revoke any reset tokens/OTPs
                        new_hashed = bcrypt.hashpw(new_pw.encode(), bcrypt.gensalt())
                        users_collection.update_one(
                            {"username": st.session_state.username},
                            {"$set": {"password": new_hashed},
                             "$unset": {"otp_hash": "", "otp_expiry": "", "reset_jwt": "", "reset_jwt_expiry": ""}}
                        )
                        # Regenerate a fresh JWT for the session so token reflects any updated claims/expiry
                        try:
                            new_token = generate_token(st.session_state.username)
                            st.session_state.jwt_token = new_token
                        except Exception:
                            # If token generation fails ignore but log
                            log_action("regenerate_jwt_after_pw_change_failed", username=st.session_state.username, status="failed")

                        st.success("Password updated successfully.")
                        log_action("change_password_success", username=st.session_state.username, status="success")
                except Exception as e:
                    st.error(f"An error occurred while changing password: {e}")
                    log_action("change_password_error", username=st.session_state.username, status="failed", meta={"error": str(e)})

    st.markdown("---")

    # Optional: show recent login / security info
    with st.expander("Security & Activity"):
      st.subheader("Security & Activity")
      last_actions = list(logs_collection.find({"username": st.session_state.username}).sort("timestamp", -1).limit(5))
      if last_actions:
          for act in last_actions:
              ts = act.get("timestamp")
              ts_str = ts.strftime("%Y-%m-%d %H:%M:%S") if isinstance(ts, datetime.datetime) else str(ts)
              st.write(f"- **{ts_str}** — {act.get('action','unknown')} — {act.get('status','')}")
      else:
          st.write("No recent activity available.")

        # -------------------------
    # Request Admin Access (for regular users)
    # -------------------------
    # Show only for normal users (not admins)
    try:
        current_role = st.session_state.get("role", None)
        if current_role == "user":
            st.markdown("---")
            st.subheader("🔐 Request Admin Access")
            # Check if there is already a pending request for this user
            existing_req = requests_collection.find_one({
                "username": st.session_state.username,
                "kind": "admin_request",
                "status": "pending"
            })
            if existing_req:
                # Show info and allow cancel
                created = existing_req.get("created_at")
                created_str = created.strftime("%Y-%m-%d %H:%M:%S") if isinstance(created, datetime.datetime) else str(created)
                st.info(f"You already have a pending admin request (created: {created_str}). An admin will review it soon.")
                colr1, colr2 = st.columns([1,3])
                with colr1:
                    if st.button("Cancel Request", key=f"cancel_admin_req_{st.session_state.username}"):
                        requests_collection.update_one(
                            {"_id": existing_req["_id"]},
                            {"$set": {"status": "cancelled", "processed_at": datetime.datetime.utcnow(), "processed_by": st.session_state.username}}
                        )
                        log_action("cancel_admin_request", username=st.session_state.username, meta={"request_id": str(existing_req["_id"])})
                        st.success("Your admin request was cancelled.")
                        st.rerun()
            else:
                st.write("If you need admin privileges, submit a request below. Admins will review and approve or reject it.")
                reason = st.text_area("Why do you need admin access? (optional)", key="admin_request_reason", help="Write a short reason so admins can decide.")
                if st.button("Request Admin Access", key=f"request_admin_{st.session_state.username}"):
                    # Insert a pending request document
                    req_doc = {
                        "username": st.session_state.username,
                        "kind": "admin_request",
                        "payload": {"requested_role": "admin", "reason": reason},
                        "status": "pending",
                        "created_at": datetime.datetime.utcnow()
                    }
                    requests_collection.insert_one(req_doc)
                    log_action("submit_admin_request", username=st.session_state.username, meta={"payload": req_doc["payload"]})
                    st.success("Admin request submitted. An administrator will review your request.")
                    st.rerun()
    except Exception as e:
        st.error(f"Could not process admin-request UI: {e}")
        log_action("admin_request_ui_error", username=st.session_state.username, status="failed", meta={"error": str(e)})


    # Provide a Logout button here as well if user wants to logout after changing password
    if st.button("Logout"):
        st.session_state.clear()
        st.session_state.page = "login"
        st.rerun()
# -----------------------------
# 🔐 Login / Signup / OTP Pages
# -----------------------------

def login_signup_page():
    apply_neon_css()
    render_title_header()
    # # --- Debug block: paste at beginning of login_signup_page() ---
    # st.markdown("## OAuth debug info (temporary)")
    # qp = st.experimental_get_query_params()
    # st.write("query params:", qp)
    # st.write("st.session_state.oauth_state (if any):", st.session_state.get("oauth_state"))
    # st.write("APP_BASE_URL (from env):", APP_BASE_URL)
    # st.write("GOOGLE_CLIENT_ID loaded?:", bool(GOOGLE_CLIENT_ID))
    # # Do NOT print GOOGLE_CLIENT_SECRET
    # st.markdown("---")
    # # --- end debug block ---


    # --- If Google redirected back with code, handle it first ---
    query_params = st.experimental_get_query_params()
    if "code" in query_params:
        # Google has redirected back with an authorization code
        code = query_params.get("code")[0]
        state = query_params.get("state", [None])[0]
        # Optional: check st.session_state['oauth_state'] vs state if you saved it earlier
        try:
            token_resp = exchange_code_for_tokens(code)
            idt = token_resp.get("id_token")
            claims = verify_google_id_token(idt)
            if claims:
                login_via_google(claims)
                # cleanup query params so user doesn't re-run token exchange on refresh
                st.experimental_set_query_params()
                st.rerun()
            else:
                st.error("Google token verification failed.")
        except Exception as e:
            st.error(f"Google sign-in failed: {e}")

    # Centered container with padding
    st.markdown("<div style='display:flex;justify-content:center;align-items:center;min-height:6vh;'>", unsafe_allow_html=True)
    col_left, col_center, col_right = st.columns([1, 1, 1])

    with col_center:
        tab1, tab2 = st.tabs([" Login", " Signup"])

        with tab1:
            username = st.text_input("Username (Email)", key="login_username")
            password = st.text_input("Password", type="password", key="login_password")

            # Traditional login/signup buttons
            col1, col2 = st.columns(2)

            with col1:
                if st.button("Login", key="login_btn", use_container_width=True):
                    if verify_user(username, password):
                        token = generate_token(username)
                        st.session_state.jwt_token = token
                        st.session_state.username = username
                        user = users_collection.find_one({"username": username})
                        st.session_state.role = user.get("role", "user")
                        st.session_state.page = "main"
                        st.rerun()
                    else:
                        st.error("Invalid credentials.")

            with col2:
                forgot_click = st.button("Forgot Password?", key="forgot_pw_btn", use_container_width=True)
                if forgot_click:
                    st.session_state.page = "otp"
                    st.session_state.otp_user = username
                    st.rerun()

            st.markdown("---")
            # --- Google Sign-In UI ---
            st.markdown("**Or sign in with Google**")
            # Generate an ephemeral state and store it to verify when Google redirects back
            if "oauth_state" not in st.session_state:
                st.session_state.oauth_state = secrets.token_urlsafe(16)
            oauth_url = make_google_oauth_url(st.session_state.oauth_state)
            # Nice button-like link (uses Google's blue for visual hint)
            st.markdown(f"""
                <a href="{oauth_url}" style="display:inline-block;padding:10px 18px;background:#4285F4;color:white;border-radius:6px;text-decoration:none;font-weight:700;">
                    Sign in with Google
                </a>
            """, unsafe_allow_html=True)

        with tab2:
            # --- Signup with username + email + OTP verification ---
            # We keep pending signup in st.session_state['pending_signup'] until OTP is verified.
            pending = st.session_state.get("pending_signup")

            if not pending:
                st.markdown("### Create a new account")
                chosen_username = st.text_input("Choose a username (display name)", key="signup_displayname")
                signup_email = st.text_input("Email (will be used to login)", key="signup_email")
                signup_password = st.text_input("Password", type="password", key="signup_password")

                if st.button("Sign up", key="signup_btn"):
                    # basic validation
                    if not chosen_username or not signup_email or not signup_password:
                        st.error("Please fill all fields (username, email and password).")
                    else:
                        email_norm = signup_email.lower().strip()
                        # check email or display name collisions
                        if users_collection.find_one({"username": email_norm}):
                            st.error("An account with that email already exists. Try logging in.")
                        elif users_collection.find_one({"display_name": chosen_username.strip()}):
                            st.error("That username/display name is already taken. Choose another.")
                        else:
                            # prepare pending signup: generate OTP + reset_jwt + expiry
                            otp = generate_otp()  # function already in your file
                            reset_jwt = generate_reset_jwt(email_norm, minutes_valid=10)
                            otp_expiry = datetime.datetime.utcnow() + datetime.timedelta(seconds=300)  # 5 minutes
                            # store pending info in session (ephemeral)
                            st.session_state["pending_signup"] = {
                                "display_name": chosen_username.strip(),
                                "email": email_norm,
                                "password": signup_password,   # ephemeral only until verified
                                "otp": otp,
                                "otp_expires_at": otp_expiry,
                                "reset_jwt": reset_jwt,
                                "otp_sent_at": datetime.datetime.utcnow(),
                                "otp_attempts": 0
                            }
                            # try sending OTP email (use your existing helper with link)
                            sent = False
                            try:
                                # send_otp_email_with_link(to_email, otp, reset_jwt, app_base_url=None)
                                # we prefer to include APP_BASE_URL so the link opens the app with ?reset_jwt=...
                                app_base = os.getenv("APP_BASE_URL", None)
                                sent = send_otp_email_with_link(email_norm, otp, reset_jwt, app_base_url=app_base)
                            except Exception as e:
                                sent = False
                                st.warning(f"Error sending OTP email: {e}")

                            if sent:
                                st.success("OTP sent to your email. Enter it below to verify and finish signup.")
                                st.rerun()
                            else:
                                # for dev: show OTP in UI only if email fails (you have similar logic in password reset)
                                st.warning("Failed to send OTP email. If this is a dev environment, OTP is shown below for testing.")
                                st.info(f"(DEV) OTP for {email_norm}: {otp}")
                                st.rerun()

            else:
                # pending signup exists -> show OTP verification UI
                st.markdown("### Verify your email to complete signup")
                st.write(f"OTP was sent to **{pending['email']}** (username: **{pending['display_name']}**)")

                now = datetime.datetime.utcnow()
                if now > pending["otp_expires_at"]:
                    st.error("OTP has expired. Click `Resend OTP` to get a new code.")
                else:
                    remaining = pending["otp_expires_at"] - now
                    mins, secs = divmod(int(remaining.total_seconds()), 60)
                    st.info(f"OTP expires in {mins}m {secs}s")

                entered_otp = st.text_input("Enter OTP", key="signup_entered_otp")
                colv1, colv2 = st.columns([1,1])

                with colv1:
                    if st.button("Verify OTP", key="verify_otp_btn"):
                        pending = st.session_state.get("pending_signup")
                        if not pending:
                            st.error("No pending signup found. Start signup again.")
                        else:
                            pending["otp_attempts"] = pending.get("otp_attempts", 0) + 1
                            if pending["otp_attempts"] > 5:
                                st.session_state.pop("pending_signup", None)
                                st.error("Maximum OTP attempts exceeded. Start signup again.")
                            elif datetime.datetime.utcnow() > pending["otp_expires_at"]:
                                st.error("OTP expired. Please resend a new OTP.")
                            else:
                                # compare OTP (plain compare — OTP stored in session)
                                if entered_otp.strip() == pending["otp"]:
                                    # create user now using existing create_user() helper
                                    ok, msg = create_user(pending["email"], pending["password"], role="user")
                                    if not ok:
                                        st.error(msg)
                                        # cleanup pending
                                        st.session_state.pop("pending_signup", None)
                                    else:
                                        # store display name if needed
                                        users_collection.update_one({"username": pending["email"]}, {"$set": {"display_name": pending["display_name"]}})
                                        st.success("Account created successfully — you are logged in.")
                                        # create token & set session like existing login flow
                                        token = generate_token(pending["email"])
                                        st.session_state.jwt_token = token
                                        st.session_state.username = pending["email"]
                                        st.session_state.role = "user"
                                        st.session_state.pop("pending_signup", None)
                                        st.rerun()
                                else:
                                    st.error("Invalid OTP. Try again.")
                                    st.session_state["pending_signup"] = pending  # persist attempt increment

                with colv2:
                    # resend OTP
                    if st.button("Resend OTP", key="resend_otp_btn"):
                        pending = st.session_state.get("pending_signup")
                        if not pending:
                            st.error("No signup pending.")
                        else:
                            # generate new otp, update expiry and send
                            new_otp = generate_otp()
                            pending["otp"] = new_otp
                            pending["otp_expires_at"] = datetime.datetime.utcnow() + datetime.timedelta(seconds=300)
                            pending["otp_sent_at"] = datetime.datetime.utcnow()
                            pending["otp_attempts"] = 0
                            st.session_state["pending_signup"] = pending
                            try:
                                sent = send_otp_email_with_link(pending["email"], new_otp, pending["reset_jwt"], app_base_url=os.getenv("APP_BASE_URL"))
                                if sent:
                                    st.success("New OTP sent to your email.")
                                else:
                                    st.warning("Failed to resend OTP by email. (DEV) OTP shown below:")
                                    st.info(f"(DEV) New OTP for {pending['email']}: {new_otp}")
                            except Exception as e:
                                st.error(f"Failed to resend OTP: {e}")

                # allow cancel
                if st.button("Cancel signup", key="cancel_signup_btn"):
                    st.session_state.pop("pending_signup", None)
                    st.rerun()

            st.markdown("---")
            # --- Google Sign-In UI ---
            st.markdown("**Or sign in with Google**")
            # Generate an ephemeral state and store it to verify when Google redirects back
            if "oauth_state" not in st.session_state:
                st.session_state.oauth_state = secrets.token_urlsafe(16)
            oauth_url = make_google_oauth_url(st.session_state.oauth_state)
            # Nice button-like link (uses Google's blue for visual hint)
            st.markdown(f"""
                <a href="{oauth_url}" style="display:inline-block;padding:10px 18px;background:#4285F4;color:white;border-radius:6px;text-decoration:none;font-weight:700;">
                    Sign in with Google
                </a>
            """, unsafe_allow_html=True)

        st.markdown("</div>", unsafe_allow_html=True)
    st.markdown("</div>", unsafe_allow_html=True)


# def login_signup_page():
#     apply_neon_css()
#     render_title_header()

#     # Centered container with padding
#     st.markdown("<div style='display:flex;justify-content:center;align-items:center;min-height:6vh;'>", unsafe_allow_html=True)
#     col_left, col_center, col_right = st.columns([1, 1, 1])

#     with col_center:
#         tab1, tab2 = st.tabs([" Login", " Signup"])

#         with tab1:
#             username = st.text_input("Username (Email)", key="login_username")
#             password = st.text_input("Password", type="password", key="login_password")

#             # Create columns for login and forgot password buttons on the same line
#             col1, col2 = st.columns(2)

#             with col1:
#                 if st.button("Login", key="login_btn", use_container_width=True):
#                     if verify_user(username, password):
#                         token = generate_token(username)
#                         st.session_state.jwt_token = token
#                         st.session_state.username = username
#                         user = users_collection.find_one({"username": username})
#                         st.session_state.role = user.get("role", "user")
#                         st.session_state.page = "main"
#                         st.rerun()
#                     else:
#                         st.error("Invalid credentials.")
#                 oauth_login_ui()

#             with col2:

#                 forgot_click = st.button("Forgot Password?", key="forgot_pw_btn", use_container_width=True)
#                 if forgot_click:
#                     st.session_state.page = "otp"
#                     st.session_state.otp_user = username
#                     st.rerun()

#         with tab2:
#             new_user = st.text_input("New Username (Email)", key="signup_username")
#             new_pass = st.text_input("New Password", type="password", key="signup_password")
#             if st.button("Signup", key="signup_btn"):
#                 ok, msg = create_user(new_user, new_pass)
#                 if ok:
#                     st.success(msg)
#                 else:
#                     st.error(msg)

#         st.markdown("</div>", unsafe_allow_html=True)
#     st.markdown("</div>", unsafe_allow_html=True)
# -----------------------------
# 🔐 Password-reset JWT helpers
# -----------------------------
def generate_reset_jwt(username, minutes_valid=10):
    payload = {
        "username": username,
        "purpose": "password_reset",
        "exp": datetime.datetime.utcnow() + datetime.timedelta(minutes=minutes_valid),
        "iat": datetime.datetime.utcnow()
    }
    # returns a string token
    token = jwt.encode(payload, JWT_SECRET, algorithm="HS256")
    return token

def decode_reset_jwt(token):
    try:
        payload = jwt.decode(token, JWT_SECRET, algorithms=["HS256"])
        # Ensure purpose is password_reset
        if payload.get("purpose") != "password_reset":
            return None
        return payload
    except Exception as e:
        # decode failure: expired or tampered
        return None


# def password_reset_page():
#     st.title("🔒 Password Reset via OTP")
#     if "otp_stage" not in st.session_state:
#         st.session_state.otp_stage = "request"
#     st.write(st.session_state)
#     if st.session_state.otp_stage == "request":
#         email = st.text_input("Enter your registered email")
#         if st.button("Send OTP"):
#             user = users_collection.find_one({"username": email})
#             if not user:
#                 st.error("User not found.")
#                 log_action("password_reset_request", username=email, status="user_not_found")
#                 return
#             otp = generate_otp()
#             hashed_otp = bcrypt.hashpw(otp.encode(), bcrypt.gensalt())
#             expiry = datetime.datetime.utcnow() + datetime.timedelta(minutes=5)
#             users_collection.update_one({"username": email},
#                 {"$set": {"otp_hash": hashed_otp, "otp_expiry": expiry}})
#             if send_otp_email(email, otp):

#                 st.success("OTP sent! Check your email.")
#                 st.session_state.otp_stage = "verify"
#                 st.session_state.otp_user = email
#                 log_action("password_reset_request", username=email, status="otp_sent")
#                 st.write(st.session_state)
#                 # Use rerun for immediate UI update
#             else:
#                 st.error("Failed to send OTP email. Please check SMTP settings or try again.") # More specific error
#                 # Option: show OTP in UI for demo if email fails
#                 st.warning(f"For demo purposes, if email failed, OTP for {email}: {otp}")
#                 st.session_state.otp_stage = "verify"
#                 st.session_state.otp_user = email
#             st.rerun()


#     elif st.session_state.otp_stage == "verify":
#         st.write("verify mein aa gye hai")
#         st.write(f"Verifying OTP for {st.session_state.otp_user}")
#         otp_input = st.text_input("Enter OTP")
#         new_pass = st.text_input("Enter New Password", type="password")
#         if st.button("Verify & Reset"):
#             user = users_collection.find_one({"username": st.session_state.otp_user})
#             if not user or "otp_hash" not in user:
#                 st.error("OTP not generated or expired. Please request a new OTP.")
#                 log_action("password_reset_verify", username=st.session_state.otp_user, status="no_otp_found")
#                 st.session_state.otp_stage = "request" # Reset stage
#                 st.session_state.otp_user = None
#                 st.rerun()
#                 return
#             if datetime.datetime.utcnow() > user["otp_expiry"]:
#                 st.error("OTP expired. Please request a new OTP.")
#                 log_action("password_reset_verify", username=st.session_state.otp_user, status="otp_expired")
#                 st.session_state.otp_stage = "request" # Reset stage
#                 st.session_state.otp_user = None
#                 st.rerun()
#                 return
#             if bcrypt.checkpw(otp_input.encode(), user["otp_hash"]):
#                 hashed = bcrypt.hashpw(new_pass.encode(), bcrypt.gensalt())
#                 users_collection.update_one({"username": st.session_state.otp_user},
#                     {"$set": {"password": hashed}, "$unset": {"otp_hash": "", "otp_expiry": ""}})
#                 st.success("Password updated! You can now login.")
#                 log_action("password_reset_verify", username=st.session_state.otp_user, status="success")
#                 st.session_state.page = "login"
#                 st.session_state.otp_stage = None # Reset stage
#                 st.session_state.otp_user = None
#                 st.rerun()
#             else:
#                 st.error("Invalid OTP.")
#                 log_action("password_reset_verify", username=st.session_state.otp_user, status="invalid_otp")


def password_reset_page():
    apply_neon_css()
    st.title("🔒 Password Reset via OTP")
    if st.button("⬅ Back to Login"):
          st.session_state.page = "login_signup"
          st.rerun()
    # Ensure otp_stage default is set only if not present
    if "otp_stage" not in st.session_state or st.session_state.otp_stage is None:
        st.session_state.otp_stage = "request"

    # Debugging aid (optional) - comment out if noisy
    # st.write("Session state:", dict(st.session_state))

    # ---------- Stage: request OTP ----------
    if st.session_state.otp_stage == "request":
        st.write("Enter your registered email to receive an OTP.")
        email = st.text_input("Registered email", key="pr_email")
        if st.button("Send OTP"):
            user = users_collection.find_one({"username": email})
            if not user:
                st.error("User not found.")
                log_action("password_reset_request", username=email, status="user_not_found")
                return
            # otp = generate_otp()
            # hashed_otp = bcrypt.hashpw(otp.encode(), bcrypt.gensalt())
            # expiry = datetime.datetime.utcnow() + datetime.timedelta(minutes=5)
            # users_collection.update_one({"username": email},
            #                             {"$set": {"otp_hash": hashed_otp, "otp_expiry": expiry}})

            # Example inside password_reset_page() when handling "Send OTP"
            otp = generate_otp()
            hashed_otp = bcrypt.hashpw(otp.encode(), bcrypt.gensalt())
            expiry = datetime.datetime.utcnow() + datetime.timedelta(minutes=5)

            # Create short-lived reset JWT (e.g. 10 minutes) and store it
            reset_jwt = generate_reset_jwt(email, minutes_valid=10)
            reset_jwt_expiry = datetime.datetime.utcnow() + datetime.timedelta(minutes=10)

            # Store both otp_hash and reset_jwt (or hash of it) in user doc
            users_collection.update_one(
                {"username": email},
                {"$set": {"otp_hash": hashed_otp, "otp_expiry": expiry, "reset_jwt": reset_jwt, "reset_jwt_expiry": reset_jwt_expiry}}
            )

            # Send email with OTP and link
            send_ok = send_otp_email_with_link(email, otp, reset_jwt, app_base_url=os.getenv("APP_BASE_URL"))

            if send_otp_email_with_link(email, otp,reset_jwt):
                st.success("OTP sent! Check your email.")
                st.session_state.otp_stage = "verify"
                st.session_state.otp_user = email
                st.rerun()
            else:
                st.error("Failed to send OTP email. Please check SMTP settings or try again.")
                # For dev/demo: optionally surface OTP if email fails
                st.warning(f"(DEV) OTP for {email}: {otp}")
                st.session_state.otp_stage = "verify"
                st.session_state.otp_user = email
                st.rerun()

        # ---------- Stage: verify OTP (only OTP input here) ----------
    elif st.session_state.otp_stage == "verify":
        st.write(f"Verifying OTP for {st.session_state.otp_user}")
        otp_input = st.text_input("Enter OTP", key="pr_otp")
        if st.button("Verify OTP"):
            user = users_collection.find_one({"username": st.session_state.otp_user})
            if not user or "otp_hash" not in user:
                st.error("OTP not generated or expired. Please request a new OTP.")
                log_action("password_reset_verify", username=st.session_state.otp_user, status="no_otp_found")
                # Reset flow to request stage
                st.session_state.otp_stage = "request"
                st.session_state.otp_user = None
                st.rerun()
                return

            # 1) Check OTP expiry
            if datetime.datetime.utcnow() > user.get("otp_expiry", datetime.datetime.utcfromtimestamp(0)):
                st.error("OTP expired. Please request a new OTP.")
                log_action("password_reset_verify", username=st.session_state.otp_user, status="otp_expired")
                # Invalidate stored otp/jwt on expiry
                users_collection.update_one({"username": st.session_state.otp_user},
                                            {"$unset": {"otp_hash": "", "otp_expiry": "", "reset_jwt": "", "reset_jwt_expiry": ""}})
                st.session_state.otp_stage = "request"
                st.session_state.otp_user = None
                st.rerun()
                return

            # 2) Ensure a reset_jwt exists in DB and hasn't expired
            reset_jwt = user.get("reset_jwt")
            reset_jwt_expiry = user.get("reset_jwt_expiry")
            if not reset_jwt or not reset_jwt_expiry:
                st.error("Reset session missing. Please request a new OTP.")
                users_collection.update_one({"username": st.session_state.otp_user},
                                            {"$unset": {"otp_hash": "", "otp_expiry": ""}})
                st.session_state.otp_stage = "request"
                st.session_state.otp_user = None
                st.rerun()
                return

            # Check reset_jwt_expiry stored in DB
            if datetime.datetime.utcnow() > reset_jwt_expiry:
                st.error("Reset session expired. Please request a new OTP.")
                log_action("password_reset_verify", username=st.session_state.otp_user, status="reset_jwt_expired")
                # Invalidate stored otp/jwt on expiry
                users_collection.update_one({"username": st.session_state.otp_user},
                                            {"$unset": {"otp_hash": "", "otp_expiry": "", "reset_jwt": "", "reset_jwt_expiry": ""}})
                st.session_state.otp_stage = "request"
                st.session_state.otp_user = None
                st.rerun()
                return

            # 3) Optionally, verify JWT signature & payload (defense-in-depth)
            payload = decode_reset_jwt(reset_jwt)
            if not payload or payload.get("username") != st.session_state.otp_user or payload.get("purpose") != "password_reset":
                st.error("Invalid reset session token. Please request a new OTP.")
                users_collection.update_one({"username": st.session_state.otp_user},
                                            {"$unset": {"otp_hash": "", "otp_expiry": "", "reset_jwt": "", "reset_jwt_expiry": ""}})
                st.session_state.otp_stage = "request"
                st.session_state.otp_user = None
                st.rerun()
                return

            # 4) Finally check the OTP itself
            if bcrypt.checkpw(otp_input.encode(), user["otp_hash"]):
                st.success("OTP verified! Please set your new password.")
                log_action("password_reset_verify", username=st.session_state.otp_user, status="success")
                # Move to reset stage (where only new password is requested)
                st.session_state.otp_stage = "reset"
                st.rerun()
            else:
                st.error("Invalid OTP.")
                log_action("password_reset_verify", username=st.session_state.otp_user, status="invalid_otp")

        # Option to re-request OTP (resend)
        st.write("---")
        if st.button("Resend OTP"):
            # Invalidate current otp and jwt
            users_collection.update_one({"username": st.session_state.otp_user},
                                        {"$unset": {"otp_hash": "", "otp_expiry": "", "reset_jwt": "", "reset_jwt_expiry": ""}})
            st.session_state.otp_stage = "request"
            # keep otp_user None so they re-enter email
            st.session_state.otp_user = None
            st.rerun()

    # ---------- Stage: reset (enter new password only) ----------
    elif st.session_state.otp_stage == "reset":
        st.write(f"Set a new password for {st.session_state.otp_user}")
        new_pass = st.text_input("Enter New Password", type="password", key="pr_newpass")
        new_pass_confirm = st.text_input("Confirm New Password", type="password", key="pr_newpass_conf")
        if st.button("Set New Password"):
            if not new_pass:
                st.error("Please enter a new password.")
                return
            if new_pass != new_pass_confirm:
                st.error("Passwords do not match.")
                return

            # Double-check user still exists
            user = users_collection.find_one({"username": st.session_state.otp_user})
            if not user:
                st.error("User no longer exists. Please request OTP again.")
                st.session_state.otp_stage = "request"
                st.session_state.otp_user = None
                st.rerun()
                return

            # # Update password and remove OTP fields
            # hashed = bcrypt.hashpw(new_pass.encode(), bcrypt.gensalt())
            # users_collection.update_one(
            #     {"username": st.session_state.otp_user},
            #     {"$set": {"password": hashed}, "$unset": {"otp_hash": "", "otp_expiry": ""}}
            # )
                        # Update password and remove OTP + reset_jwt fields (single-use)
            hashed = bcrypt.hashpw(new_pass.encode(), bcrypt.gensalt())
            users_collection.update_one(
                {"username": st.session_state.otp_user},
                {"$set": {"password": hashed},
                 "$unset": {"otp_hash": "", "otp_expiry": "", "reset_jwt": "", "reset_jwt_expiry": ""}}
            )

            st.success("Password updated! You can now log in.")
            log_action("password_reset_complete", username=st.session_state.otp_user, status="success")

            # Clear otp state and redirect to login
            st.session_state.otp_stage = None
            st.session_state.otp_user = None
            st.session_state.page = "login"
            st.rerun()

        # Option to cancel and go back to login
        if st.button("Cancel"):
            st.session_state.otp_stage = None
            st.session_state.otp_user = None
            st.session_state.page = "login"
            st.rerun()

    else:
        # Fallback safety - reset state
        st.warning("Unknown OTP stage. Resetting flow.")
        st.session_state.otp_stage = "request"
        st.session_state.otp_user = None
        st.rerun()




# -----------------------------
# Auto-cleanup files older than 30 days
# -----------------------------
def cleanup_expired_files():
    cutoff = datetime.datetime.utcnow() - datetime.timedelta(days=30)
    result = files_collection.delete_many({"uploaded_at": {"$lt": cutoff}})
    if result.deleted_count > 0:
        log_action("cleanup_files", username="system", meta={"deleted": result.deleted_count})
# cleanup_expired_files()  # run at app start/load - Better to schedule this externally or less often


# -----------------------------
# Admin: analytics & charts
# -----------------------------

# Replace your existing analytics_charts() with this enhanced version
import plotly.graph_objects as go
import plotly.express as px

def analytics_charts():
    st.subheader("Analytics & Trends")

    # Refresh button (forces re-run)
    if st.button("Refresh analytics", key="refresh_analytics_btn"):
        st.rerun()

    # --- Fetch data ---
    # tune limits if you have very large collections
    users_cursor = users_collection.find({}, {"password": 0}).sort("created_at", -1).limit(5000)
    hist_cursor = history_collection.find({}).sort("timestamp", -1).limit(10000)

    users_df = df_from_cursor(users_cursor)
    hist_df = df_from_cursor(hist_cursor)

    # Normalize datetimes
    if not users_df.empty and 'created_at' in users_df.columns:
        users_df['created_at'] = pd.to_datetime(users_df['created_at'], errors='coerce')
    if not hist_df.empty and 'timestamp' in hist_df.columns:
        hist_df['timestamp'] = pd.to_datetime(hist_df['timestamp'], errors='coerce')

    # # Debug info: counts & most recent history entries
    # st.write(f"Users fetched: {len(users_df)}")
    # st.write(f"History rows fetched: {len(hist_df)}")
    # if not hist_df.empty:
    #     try:
    #         st.write("Most recent history entries (debug):")
    #         st.dataframe(hist_df.sort_values('timestamp', ascending=False).head(5)[[c for c in ['timestamp','username','model','prompt','event','action','type'] if c in hist_df.columns]])
    #     except Exception:
    #         pass

    # --- Top row: two columns ---
    col_left, col_right = st.columns([2, 2])

    # LEFT: User signups (time series) & Daily logins
    with col_left:
        st.markdown("### User Signups")
        if users_df.empty or 'created_at' not in users_df.columns or users_df['created_at'].isnull().all():
            st.info("No signup timestamp data available.")
        else:
            signups = users_df.dropna(subset=['created_at']).copy()
            signups['date'] = signups['created_at'].dt.date
            signups_daily = signups.groupby('date').size().reset_index(name='count')
            fig = px.line(signups_daily, x='date', y='count', title="User Signups Over Time")
            fig.update_layout(xaxis_title="Date", yaxis_title="Signups")
            st.plotly_chart(fig, use_container_width=True)

        # --- Daily Logins (logs_collection-specific replacement) ---
        st.markdown("### Daily Logins (from logs_collection)")

        # Quick guard: ensure logs_collection exists
        if 'logs_collection' not in globals():
            st.info("No logs_collection found in globals. Make sure logs_collection is created and available.")
        else:
            # Option: count only successful logins or include failed attempts
            status_choice = st.selectbox("Count which logins?", ["success only", "all attempts"], index=0)
            count_only_success = (status_choice == "success only")

            # Fetch recent logs (tune limit as needed)
            try:
                cursor = logs_collection.find({"action": "login"}).sort("timestamp_str", -1).limit(20000)
                logs_df = df_from_cursor(cursor)  # reuse your helper that converts cursor -> DataFrame
            except Exception as e:
                st.error(f"Failed to read logs_collection: {e}")
                logs_df = pd.DataFrame()

            if logs_df.empty:
                st.info("No login entries found in logs_collection.")
            else:
                # Normalize timestamp_str -> timestamp
                # Try multiple possible timestamp columns if present
                if 'timestamp' in logs_df.columns and logs_df['timestamp'].notnull().any():
                    logs_df['ts'] = pd.to_datetime(logs_df['timestamp'], errors='coerce')
                else:
                    # primary is timestamp_str per your sample
                    if 'timestamp_str' in logs_df.columns:
                        # handle possible formats; fallback to errors='coerce'
                        logs_df['ts'] = pd.to_datetime(logs_df['timestamp_str'], format="%Y-%m-%d %H:%M:%S", errors='coerce')
                    else:
                        # fallback: try any string-looking column
                        strcols = [c for c in logs_df.columns if logs_df[c].dtype == object]
                        logs_df['ts'] = None
                        for c in ['timestamp_str','time','created_at'] + strcols:
                            if c in logs_df.columns:
                                tmp = pd.to_datetime(logs_df[c], errors='coerce')
                                if tmp.notnull().any():
                                    logs_df['ts'] = tmp
                                    break

                # Filter by status if desired
                if count_only_success and 'status' in logs_df.columns:
                    filtered = logs_df[logs_df['status'].astype(str).str.lower() == 'success'].copy()
                elif count_only_success and 'status' not in logs_df.columns:
                    st.warning("No `status` column present; counting all login attempts instead.")
                    filtered = logs_df.copy()
                else:
                    filtered = logs_df.copy()

                # Drop rows missing parsed timestamps
                if 'ts' in filtered.columns:
                    filtered = filtered.dropna(subset=['ts'])
                else:
                    filtered['ts'] = pd.to_datetime(filtered.get('timestamp_str', None), errors='coerce')
                    filtered = filtered.dropna(subset=['ts'])

                if filtered.empty:
                    st.info("No login rows with valid timestamps found after filtering.")
                else:
                    # Create date column and aggregate
                    filtered['date'] = filtered['ts'].dt.date
                    daily = filtered.groupby('date').size().reset_index(name='logins').sort_values('date')

                    # Plot bar chart
                    fig_logins = px.bar(daily, x='date', y='logins', title="Daily Logins (from logs_collection)",
                                        labels={'date': 'Date', 'logins': 'Count'})
                    fig_logins.update_layout(xaxis_tickangle=-45, height=320, margin=dict(t=40, b=80))
                    st.plotly_chart(fig_logins, use_container_width=True)


        # st.markdown("### Daily Logins")
        # # Detect likely 'login' events in history (various possible column names)
        # login_candidate_cols = [c for c in ['event','action','type','activity'] if c in hist_df.columns]
        # if hist_df.empty or not login_candidate_cols:
        #     st.info("No login/event column detected in history to compute daily logins.")
        # else:
        #     # try to detect login-like rows
        #     login_col = login_candidate_cols[0]
        #     # consider values that contain 'login' case-insensitive
        #     hist_df['__event_str__'] = hist_df[login_col].astype(str).str.lower()
        #     login_rows = hist_df[hist_df['__event_str__'].str.contains('login', na=False)]
        #     if login_rows.empty:
        #         # maybe login recorded as 'user_login' or 'auth' etc. try common markers
        #         login_rows = hist_df[hist_df['__event_str__'].str.contains('auth|signin|sign_in|logged', na=False)]
        #     if login_rows.empty or 'timestamp' not in login_rows.columns:
        #         st.info("No explicit login events found in history.")
        #     else:
        #         login_rows = login_rows.dropna(subset=['timestamp']).copy()
        #         login_rows['date'] = login_rows['timestamp'].dt.date
        #         login_daily = login_rows.groupby('date').size().reset_index(name='logins')
        #         fig2 = px.bar(login_daily, x='date', y='logins', title="Daily Logins")
        #         fig2.update_layout(xaxis_title="Date", yaxis_title="Logins")
        #         st.plotly_chart(fig2, use_container_width=True)

    # RIGHT: Model popularity pie & Language popularity pie
    with col_right:
        # Model popularity: big pie with legend below (horizontal)
        st.markdown("### Model Popularity")
        if 'model' in hist_df.columns and not hist_df['model'].isnull().all():
            model_counts = hist_df['model'].fillna('unknown').value_counts().reset_index()
            model_counts.columns = ['model','count']
            # ensure readable order
            model_counts = model_counts.sort_values('count', ascending=False)

            figm = px.pie(model_counts, values='count', names='model',
                          title="Model Popularity",
                          hole=0.0)  # set hole>0 for donut look if desired
            # make labels show inside the slices (percent only) and put legend below
            figm.update_traces(textposition='inside', textinfo='percent', hoverinfo='label+value+percent', sort=False)

            figm.update_layout(
                legend=dict(orientation='h', y=-0.15, x=0.5, xanchor='center', font=dict(size=11)),
                margin=dict(l=10, r=10, t=40, b=60),
                height=420,
                paper_bgcolor='rgba(0,0,0,0)',  # transparent so dashboard background shows
                plot_bgcolor='rgba(0,0,0,0)'
            )
            st.plotly_chart(figm, use_container_width=True)
        else:
            st.info("No model usage data available in history.")

        # st.markdown("### Model Popularity")
        # if 'model' in hist_df.columns and not hist_df['model'].isnull().all():
        #     model_counts = hist_df['model'].fillna('unknown').value_counts().reset_index()
        #     model_counts.columns = ['model','count']
        #     figm = px.pie(model_counts, values='count', names='model', title="Model Popularity")
        #     st.plotly_chart(figm, use_container_width=True)
        # else:
        #     st.info("No model usage data available in history.")

        st.markdown("### Language Popularity")
        # language column candidates
        lang_cols = [c for c in ['language','lang','code_language','user_language'] if c in hist_df.columns]
        if lang_cols:
            lang_col = lang_cols[0]
            lang_counts = hist_df[lang_col].fillna('unknown').value_counts().reset_index()
            lang_counts.columns = ['language','count']
            if lang_counts['count'].sum() == 0:
                st.info("No language information found in history.")
            else:
                fig_lang = px.pie(lang_counts, values='count', names='language', title="Language Popularity")
                st.plotly_chart(fig_lang, use_container_width=True)
        else:
            st.info("No language column detected in history records.")

    st.markdown("---")

    # --- Middle row: two charts side by side (Activity breakdown + Avg response time) ---
    mid_col1, mid_col2 = st.columns(2)

    with mid_col1:
        st.markdown("### Activity Breakdown (by kind)")
        # infer activity kind: candidates 'action','event','type'
        kind_cols = [c for c in ['mode','event','type','intent'] if c in hist_df.columns]
        if kind_cols:
            kind_col = kind_cols[0]
            kinds = hist_df[kind_col].fillna('unknown').value_counts().reset_index()
            kinds.columns = ['kind','count']
            figk = px.bar(kinds, x='kind', y='count', title="Activity by Kind")
            figk.update_layout(xaxis_title="Kind", yaxis_title="Count")
            st.plotly_chart(figk, use_container_width=True)
        else:
            st.info("No activity kind column detected to show breakdown.")

    with mid_col2:
        st.markdown("### Avg Response Time per Model")
        if 'response_time_ms' in hist_df.columns and 'model' in hist_df.columns:
            hist_df['response_time_ms'] = pd.to_numeric(hist_df['response_time_ms'], errors='coerce')
            avg_rt = hist_df.dropna(subset=['response_time_ms']).groupby('model')['response_time_ms'].mean().reset_index().sort_values('response_time_ms', ascending=False)
            if not avg_rt.empty:
                figrt = px.bar(avg_rt, x='model', y='response_time_ms', title="Avg Response Time (ms) per Model")
                st.plotly_chart(figrt, use_container_width=True)
            else:
                st.info("No numeric response_time_ms values found.")
        else:
            st.info("No response time data or model column available.")

    st.markdown("---")

    # --- Bottom: Double bar chart for generations vs explanations per day ---
    st.markdown("### Generations vs Explanations (Daily)")
    # heuristic: look for columns that indicate 'generation' or 'explanation' events
    # possible indicators: 'intent', 'task', 'action', 'event', 'type', or prompt tags
    kind_candidates = [c for c in ['mode','event','type','intent','task'] if c in hist_df.columns]
    if hist_df.empty or not kind_candidates or 'timestamp' not in hist_df.columns:
        st.info("Insufficient history data to create the generations vs explanations chart.")
    else:
        kc = kind_candidates[0]
        # map values to two buckets: generation and explanation
        def map_gen_ex(val):
            s = str(val).lower()
            if any(x in s for x in ['generate','generation','gen','create','synthesize','completion']):
                return 'generation'
            if any(x in s for x in ['explain','explanation','describe','clarify','why','walkthrough']):
                return 'explanation'
            # some systems set specific tags, try common ones
            if 'code-explain' in s or 'explain-code' in s:
                return 'explanation'
            if 'code-generate' in s or 'generate-code' in s:
                return 'generation'
            return 'other'

        temp = hist_df.dropna(subset=['timestamp']).copy()
        temp['kind_mapped'] = temp[kc].apply(map_gen_ex)
        temp['date'] = temp['timestamp'].dt.date

        # only keep the two relevant categories + others aggregated as 'other' (optional)
        daily = temp.groupby(['date','kind_mapped']).size().unstack(fill_value=0).reset_index()
        # ensure columns exist
        if 'generation' not in daily.columns:
            daily['generation'] = 0
        if 'explanation' not in daily.columns:
            daily['explanation'] = 0

        # sort by date
        daily = daily.sort_values('date')
        if daily.empty:
            st.info("No generation/explanation events detected in history.")
        else:
            # grouped bar chart
            fig_ge = go.Figure()
            fig_ge.add_trace(go.Bar(x=daily['date'], y=daily['generation'], name='Generations'))
            fig_ge.add_trace(go.Bar(x=daily['date'], y=daily['explanation'], name='Explanations'))
            fig_ge.update_layout(barmode='group', xaxis_title='Date', yaxis_title='Count', title='Generations vs Explanations per Day')
            st.plotly_chart(fig_ge, use_container_width=True)

    st.markdown("---")

    # --- Activity heatmap as last visual ---
    if not hist_df.empty and 'timestamp' in hist_df.columns:
        h = hist_df.dropna(subset=['timestamp']).copy()
        if not h.empty:
            h['hour'] = h['timestamp'].dt.hour
            h['day'] = h['timestamp'].dt.day_name()
            value_col = 'prompt' if 'prompt' in h.columns else (h.columns[0] if len(h.columns)>0 else None)
            if value_col:
                heat = h.pivot_table(index='day', columns='hour', values=value_col, aggfunc='count').fillna(0)
                days_order = ['Monday','Tuesday','Wednesday','Thursday','Friday','Saturday','Sunday']
                existing_days = [d for d in days_order if d in heat.index]
                if existing_days:
                    heat = heat.reindex(existing_days).fillna(0)
                if heat.sum().sum() > 0:
                    figh = go.Figure(data=go.Heatmap(z=heat.values, x=heat.columns, y=heat.index, colorscale='Viridis'))
                    figh.update_layout(title="Activity Heatmap (Day vs Hour)", xaxis_title="Hour", yaxis_title="Day")
                    st.plotly_chart(figh, use_container_width=True)
                else:
                    st.info("Not enough history data to generate Activity Heatmap.")
            else:
                st.info("No suitable column found to build heatmap.")
    else:
        st.info("No history data available for analytics.")
    # Debug: show the most recent login rows (timestamp, username, status, meta)

    # # Debug info: counts & most recent history entries
    st.write(f"Users fetched: {len(users_df)}")
    st.write(f"History rows fetched: {len(hist_df)}")
    preview_cols = []
    for c in ['ts', 'timestamp_str', 'username', 'status', 'meta', 'message', 'msg']:
        if c in filtered.columns:
            preview_cols.append(c)
    preview = filtered.sort_values('ts', ascending=False).head(12)[preview_cols]
    st.markdown("**Recent login entries (debug):**")
    st.dataframe(preview)

# import plotly.graph_objects as go # Import for Heatmap
# def analytics_charts():
#     st.subheader("Analytics & Trends")

#     # Users over time
#     users_df = df_from_cursor(users_collection.find({}, {"password":0}))
#     if users_df.empty:
#         st.info("No users yet")
#     else:
#         users_df['created_at'] = pd.to_datetime(users_df['created_at'])
#         users_daily = users_df.groupby(users_df['created_at'].dt.date).size().reset_index(name='count')
#         fig = px.line(users_daily, x='created_at', y='count', title="User Signups Over Time")
#         fig.update_layout(xaxis_title="Date", yaxis_title="Signups")
#         st.plotly_chart(fig, use_container_width=True)

#     # Model usage from history
#     hist_df = df_from_cursor(history_collection.find({}))
#     # Ensure 'model' column exists before trying to plot
#     if not hist_df.empty and 'model' in hist_df.columns and not hist_df['model'].isnull().all():
#         model_counts = hist_df['model'].value_counts().reset_index()
#         model_counts.columns = ['model','count']
#         fig2 = px.pie(model_counts, values='count', names='model', title="Model Popularity")
#         st.plotly_chart(fig2, use_container_width=True)

#         # Average response time if recorded in history as response_time_ms
#         if 'response_time_ms' in hist_df.columns and not hist_df['response_time_ms'].isnull().all():
#             avg_rt = hist_df.groupby('model')['response_time_ms'].mean().reset_index()
#             fig3 = px.bar(avg_rt, x='model', y='response_time_ms', title="Avg Response Time (ms) per Model")
#             st.plotly_chart(fig3, use_container_width=True)
#         else:
#              st.info("No response time data available in history.")
#     else:
#         st.info("No model usage data in history yet.")


#     # Activity heatmap: day vs hour
#     if not hist_df.empty and 'timestamp' in hist_df.columns and not hist_df['timestamp'].isnull().all():
#         hist_df['timestamp'] = pd.to_datetime(hist_df['timestamp'])
#         hist_df['hour'] = hist_df['timestamp'].dt.hour
#         hist_df['day'] = hist_df['timestamp'].dt.day_name()
#         heat = hist_df.pivot_table(index='day', columns='hour', values='prompt', aggfunc='count').fillna(0)
#         # Reorder days
#         days_order = ['Monday','Tuesday','Wednesday','Thursday','Friday','Saturday','Sunday']
#         # Reindex only if all days are not present to avoid errors
#         if all(day in heat.index for day in days_order):
#              heat = heat.reindex(days_order).fillna(0)
#         elif any(day in heat.index for day in days_order): # Reindex if some days are present
#              existing_days = [day for day in days_order if day in heat.index]
#              heat = heat.reindex(existing_days).fillna(0)
#         # Otherwise, keep the index as is if none of the standard days are found (e.g., empty history)


#         if not heat.empty: # Only create heatmap if there is data after reindexing
#             figh = go.Figure(data=go.Heatmap(z=heat.values, x=heat.columns, y=heat.index, colorscale='Viridis'))
#             figh.update_layout(title="Activity Heatmap (Day vs Hour)", xaxis_title="Hour", yaxis_title="Day")
#             st.plotly_chart(figh, use_container_width=True)
#         else:
#              st.info("Not enough history data to generate Activity Heatmap.")

#     else:
#         st.info("No history data available for analytics.")


# -----------------------------
# Admin approve/reject requests
# -----------------------------
def admin_requests_panel():
    st.subheader("Approve / Reject Requests")
    # requests schema simple: {username, kind, payload, status: pending/approved/rejected, created_at}
    pending = list(requests_collection.find({"status":"pending"}).sort("created_at",-1))
    if not pending:
        st.info("No pending requests")
    else:
        for req in pending:
            # Convert ObjectId to string for consistent keying
            req_id_str = str(req['_id'])
            with st.expander(f"{req.get('kind','request')} from {req.get('username','unknown')} - {req.get('created_at').strftime('%Y-%m-%d %H:%M:%S') if req.get('created_at') else 'N/A'}"):
                st.write(req.get('payload', {}))
                c1, c2, c3 = st.columns([1,1,1])
                if c1.button("Approve", key=f"approve_{req_id_str}"):
                    requests_collection.update_one({"_id": req["_id"]}, {"$set":{"status":"approved","processed_at": datetime.datetime.utcnow(), "processed_by": st.session_state.username}})
                    log_action("approve_request", username=st.session_state.username, meta={"request_id": req_id_str, "kind": req.get('kind')})
                    st.success("Approved")
                    st.rerun()
                if c2.button("Reject", key=f"reject_{req_id_str}"):
                    requests_collection.update_one({"_id": req["_id"]}, {"$set":{"status":"rejected","processed_at": datetime.datetime.utcnow(), "processed_by": st.session_state.username}})
                    log_action("reject_request", username=st.session_state.username, meta={"request_id": req_id_str, "kind": req.get('kind')})
                    st.warning("Rejected")
                    st.rerun()
                if c3.button("Mark as Pending", key=f"pend_{req_id_str}"):
                    requests_collection.update_one({"_id": req["_id"]}, {"$set":{"status":"pending"}})
                    log_action("reset_request_to_pending", username=st.session_state.username, meta={"request_id": req_id_str, "kind": req.get('kind')})
                    st.info("Reset to pending")
                    st.rerun()

# -----------------------------
# Logs viewer
# -----------------------------
def logs_viewer():
    st.subheader("System Logs")
    col1, col2, col3 = st.columns([3,2,2])
    with col1:
        search_user = st.text_input("Filter by username", key="log_search_user")
    with col2:
        start_date = st.date_input("From", value=(datetime.datetime.utcnow() - datetime.timedelta(days=30)).date(), key="log_start_date")
    with col3:
        end_date = st.date_input("To", value=datetime.datetime.utcnow().date(), key="log_end_date")

    q = {}
    if search_user:
        q["username"] = {"$regex": search_user, "$options":"i"}
    # Correctly format date range query for MongoDB
    start_datetime = datetime.datetime.combine(start_date, datetime.time.min)
    end_datetime = datetime.datetime.combine(end_date, datetime.time.max)
    q["timestamp"] = {"$gte": start_datetime, "$lte": end_datetime}

    # Note: small safety: limit results for performance
    logs_cursor = logs_collection.find(q).sort("timestamp",-1).limit(500)
    logs_df = df_from_cursor(logs_cursor)
    if logs_df.empty:
        st.info("No logs found matching criteria")
    else:
        # Format timestamp
        if 'timestamp' in logs_df.columns:
            logs_df['timestamp'] = pd.to_datetime(logs_df['timestamp'])
            logs_df['timestamp_str'] = logs_df['timestamp'].dt.strftime('%Y-%m-%d %H:%M:%S')
        # Define columns to display, ensuring they exist
        display_cols = ['timestamp_str','action','username','status'] + [c for c in logs_df.columns if c not in ['timestamp','_id','action','username','status','timestamp_str']]
        display_cols = [col for col in display_cols if col in logs_df.columns] # Filter existing columns
        st.dataframe(logs_df[display_cols], height=400)
        csv = logs_df.to_csv(index=False).encode()
        st.download_button("Download Logs CSV", csv, file_name=f"codegenie_logs_{datetime.datetime.utcnow().strftime('%Y%m%d')}.csv")

# -----------------------------
# User management (promote, suspend, delete)
# -----------------------------
def user_management_panel():
    st.subheader("User Management")
    search = st.text_input("Search username (case-insensitive)", key="user_mgmt_search")
    role_filter = st.selectbox("Filter by role", options=["all","user","admin"], index=0, key="user_mgmt_role_filter")
    q = {}
    if search:
        q["username"] = {"$regex": search, "$options":"i"}
    if role_filter != "all":
        q["role"] = role_filter

    users = list(users_collection.find(q, {"password": 0}).sort("created_at",-1).limit(500))
    if not users:
        st.info("No users matched.")
        return

    for u in users:
        # Use ObjectId string as unique key
        user_id_str = str(u['_id'])
        colA, colB, colC, colD = st.columns([3,1,1,1])
        with colA:
            st.markdown(f"**{u.get('username', 'N/A')}**")
            st.caption(f"Role: {u.get('role','user')} · Suspended: {u.get('suspended', False)} · Joined: {u.get('created_at').strftime('%Y-%m-%d') if u.get('created_at') else 'N/A'}")
        with colB:
            # Prevent changing own role
            if u.get('username') != st.session_state.username:
                if st.button("Promote to Admin" if u.get('role') != 'admin' else "Revoke Admin", key=f"prom_{user_id_str}"):
                    new_role = "admin" if u.get('role') != 'admin' else "user"
                    users_collection.update_one({"_id": u["_id"]}, {"$set": {"role":new_role}})
                    log_action("change_user_role", username=st.session_state.username, meta={"target": u.get('username'), "new_role": new_role})
                    st.success(f"Role changed to {new_role}")
                    st.rerun()
            else:
                st.caption("Cannot change own role")
        with colC:
            if st.button("Suspend" if not u.get('suspended',False) else "Unsuspend", key=f"sus_{user_id_str}"):
                new_suspended_status = not u.get('suspended',False)
                users_collection.update_one({"_id": u["_id"]}, {"$set": {"suspended": new_suspended_status}})
                log_action("toggle_suspend", username=st.session_state.username, meta={"target": u.get('username'), "suspended": new_suspended_status})
                st.success("Updated suspend status")
                st.rerun()
        with colD:
             # Prevent deleting own account
            if u.get('username') != st.session_state.username:
                if st.button("Delete", key=f"del_{user_id_str}"):
                     # Add a confirmation step
                    st.session_state[f'confirm_delete_user_{user_id_str}'] = True
            else:
                st.caption("Cannot delete self")

            # Confirmation logic for delete
            if st.session_state.get(f'confirm_delete_user_{user_id_str}', False):
                 st.warning(f"Are you sure you want to delete user {u.get('username', 'N/A')}? This action is irreversible.")
                 col_confirm_del1, col_confirm_del2 = st.columns(2)
                 with col_confirm_del1:
                     if st.button("Yes, Delete", key=f"confirm_del_yes_{user_id_str}", type="primary"):
                         users_collection.delete_one({"_id": u["_id"]})
                         log_action("delete_user", username=st.session_state.username, meta={"target": u.get('username')})
                         st.success("User deleted.")
                         del st.session_state[f'confirm_delete_user_{user_id_str}'] # Clear confirmation state
                         st.rerun()
                 with col_confirm_del2:
                     if st.button("Cancel", key=f"confirm_del_cancel_{user_id_str}"):
                         del st.session_state[f'confirm_delete_user_{user_id_str}'] # Clear confirmation state
                         st.rerun()


# -----------------------------
# Export helpers
# -----------------------------
# Try importing reportlab for PDF export (optional)
REPORTLAB_AVAILABLE = False
try:
    from reportlab.lib.pagesizes import letter
    from reportlab.pdfgen import canvas
    import io
    REPORTLAB_AVAILABLE = True
except ImportError:
    pass # reportlab is not installed

def export_collection_to_csv(collection, filename="export.csv", query={}):
    df = df_from_cursor(collection.find(query))
    if df.empty:
        st.info(f"No data to export from {collection.name}")
        return False # Indicate no data
    csv = df.to_csv(index=False).encode()
    st.download_button(label=f"Download {collection.name} CSV", data=csv, file_name=filename, mime="text/csv")
    return True # Indicate data was exported

# -----------------------------
# Notifications panel
# -----------------------------
def notifications_panel():
    st.subheader("Notifications")
    # New users in last 24h
    cutoff_24h = datetime.datetime.utcnow() - datetime.timedelta(hours=24)
    new_users_count = users_collection.count_documents({"created_at": {"$gte": cutoff_24h}})
    st.metric("New users (last 24h)", new_users_count)

    # High activity users (e.g., top 5 by action count in last 7 days)
    cutoff_7d = datetime.datetime.utcnow() - datetime.timedelta(days=7)
    high_activity_users = list(history_collection.aggregate([
        {"$match": {"timestamp": {"$gte": cutoff_7d}}}, # Filter by last 7 days
        {"$group":{"_id":"$username","count":{"$sum":1}}},
        {"$sort":{"count":-1}},
        {"$limit":5} # Top 5
    ]))
    st.write("Top active users (last 7 days):")
    if high_activity_users:
        for h in high_activity_users:
            st.write(f"- **{h['_id']}**: {h['count']} actions")
    else:
        st.write("No recent activity.")

    # Recent reviews (last 7 days)
    recent_reviews = list(reviews_collection.find({"timestamp": {"$gte": cutoff_7d}}).sort("timestamp", -1).limit(10))
    st.write("Recent Reviews (last 7 days):")
    if recent_reviews:
        for r in recent_reviews:
             st.write(f"**{r.get('username', 'N/A')}** ⭐ {r.get('rating', 'N/A')} ({r.get('timestamp').strftime('%Y-%m-%d') if r.get('timestamp') else 'N/A'}): {r.get('review', 'No review text')[:100]}...") # Truncate review
    else:
        st.write("No recent reviews.")

# -----------------------------
# File Management
# -----------------------------
def file_upload_panel():
    st.subheader("File Management")
    st.markdown("Upload files (images, text, csv, code). Files older than 30 days are auto-removed (this cleanup runs periodically).")
    uploaded = st.file_uploader("Drag & drop or click to upload", accept_multiple_files=True, type=None)
    if uploaded:
        for f in uploaded:
            content = f.read()
            b64 = base64.b64encode(content).decode()
            files_collection.insert_one({
                "filename": f.name,
                "content_b64": b64,
                "size": len(content),
                "mimetype": f.type if hasattr(f,"type") else "application/octet-stream",
                "owner": st.session_state.username,
                "uploaded_at": datetime.datetime.utcnow(),
                "version_of": None # Placeholder for versioning feature
            })
            log_action("upload_file", username=st.session_state.username, meta={"filename": f.name, "size": len(content)})
            st.success(f"Uploaded {f.name}")
    # List files
    files = list(files_collection.find({}).sort("uploaded_at",-1).limit(200))
    if not files:
        st.info("No files uploaded yet.")
    else:
        st.write("Files:")
        for fi in files:
            # Use ObjectId string as unique key
            file_id_str = str(fi['_id'])
            c1, c2, c3 = st.columns([3,1,1])
            with c1:
                st.markdown(f"**{fi.get('filename', 'N/A')}** — owner: {fi.get('owner','-')} — {fi.get('size', 0)} bytes — {fi.get('uploaded_at').strftime('%Y-%m-%d') if fi.get('uploaded_at') else 'N/A'}")
                # Preview small images
                if fi.get('mimetype', '').startswith("image") and fi.get('size', 0) < 2_000_000:
                    try:
                        img = Image.open(io.BytesIO(base64.b64decode(fi['content_b64'])))
                        st.image(img, width=150) # Smaller width
                    except Exception:
                         st.write("Could not display image preview.")
                else:
                    # For text/csv show first 500 chars
                    try:
                        data = base64.b64decode(fi['content_b64'])
                        # Attempt different encodings
                        text_preview = ""
                        for encoding in ['utf-8', 'latin-1']:
                            try:
                                text_preview = data[:1000].decode(encoding, errors='ignore')
                                break # Success
                            except:
                                pass # Try next encoding
                        if text_preview:
                            st.code(text_preview[:500], language='text')
                        else:
                            st.write("Could not decode file content for preview.")

                    except Exception:
                        st.write("Preview not available")
            with c2:
                # Download button needs to be within the loop and linked to the specific file
                data_to_download = base64.b64decode(fi['content_b64'])
                st.download_button(
                    label="Download",
                    data=data_to_download,
                    file_name=fi.get('filename', 'download'),
                    key=f"dl_{file_id_str}",
                    mime=fi.get('mimetype', 'application/octet-stream')
                )
            with c3:
                if st.button("Delete", key=f"delf_{file_id_str}"):
                    files_collection.delete_one({"_id": fi['_id']})
                    log_action("delete_file", username=st.session_state.username, meta={"filename": fi.get('filename')})
                    st.success("Deleted")
                    st.rerun()

def admin_panel():
    st.markdown("""
    <style>
    /* Target streamlit tab buttons */
    div[data-baseweb="tab-list"] button {
        font-size: 1.2rem !important;       /* Bigger text */
        padding: 14px 28px !important;      /* More spacing */
        height: 40px !important;            /* More height */
        border-radius: 8px !important;
        margin-up:10px !important;
    }

    /* Active tab styling */
    div[data-baseweb="tab-list"] button[aria-selected="true"] {
        background: linear-gradient(90deg, #00eaff, #6f00ff) !important;
        color: white !important;
        font-weight: bold !important;
        border: none !important;
    }

    /* Hover effect */
    div[data-baseweb="tab-list"] button:hover {
        background: rgba(255, 255, 255, 0.12) !important;
        border-radius: 8px !important;
    }
    .stTabs {
            margin-top: 5px !important;   /* adjust this value as needed */
        }

        /* Specific Streamlit structure — ensures spacing applies reliably */
    div[data-baseweb="tab-list"] {
        margin-top: 5px !important;   /* do not remove, fixes some themes */
    }
    </style>
""", unsafe_allow_html=True)

    # Ensure user is admin
    if st.session_state.get("role") != "admin":
        st.error("You do not have administrator privileges to access this page.")
        log_action("access_denied_admin_panel", username=st.session_state.username)
        return

    apply_neon_css() # Apply neon CSS for admin panel (optional, or apply site-wide)

    st.markdown("<div class='neon-box' style='padding: 20px; border-radius: 10px; box-shadow: 0 0 10px #00FFFF;'><h2 style='text-align:center; color: #00FFFF;'>👑 CodeGenie Admin Control Center</h2></div>", unsafe_allow_html=True)

    tabs = st.tabs(["Dashboard","Analytics","Requests","Users","Logs","Reviews","Files","Notifications","Exports"])

    with tabs[0]:
        st.header("Quick Overview")
        col1, col2, col3, col4 = st.columns(4)
        col1.metric("Total Users", users_collection.count_documents({}))
        col2.metric("Total Actions", history_collection.count_documents({}))
        col3.metric("Total Reviews", reviews_collection.count_documents({}))
        col4.metric("Pending Requests", requests_collection.count_documents({"status":"pending"}))

        st.markdown("---")
        st.subheader("Recent activity (last 100 entries)")
        recent = list(history_collection.find({}).sort("timestamp",-1).limit(100))
        if recent:
            hist_df = df_from_cursor(recent)
            if 'timestamp' in hist_df.columns:
                hist_df['timestamp'] = pd.to_datetime(hist_df['timestamp']).dt.strftime('%Y-%m-%d %H:%M:%S')
            # Select columns for display
            display_cols = ['timestamp', 'username', 'mode', 'model', 'language']
            display_cols = [col for col in display_cols if col in hist_df.columns] # Ensure columns exist
            st.dataframe(hist_df[display_cols].head(100))
        else:
            st.info("No recent activity")

    with tabs[1]:
        analytics_charts()

    with tabs[2]:
        admin_requests_panel()

    with tabs[3]:
        user_management_panel()

    with tabs[4]:
        logs_viewer()

    with tabs[5]:
        st.header("Reviews")
        revs = list(reviews_collection.find().sort("timestamp",-1).limit(300))
        if not revs:
            st.info("No reviews")
        else:
            for r in revs:
                # Use ObjectId string as key
                review_id_str = str(r['_id'])
                st.markdown(f"**{r.get('username', 'N/A')}** ⭐ {r.get('rating', 'N/A')} — {r.get('timestamp').strftime('%Y-%m-%d %H:%M:%S') if r.get('timestamp') else 'N/A'}")
                st.write(r.get('review', ''))
                if st.button("Delete Review", key=f"delrev_{review_id_str}"):
                    reviews_collection.delete_one({"_id": r["_id"]})
                    log_action("delete_review", username=st.session_state.username, meta={"review_id": review_id_str})
                    st.success("Review deleted.")
                    st.rerun()
                st.markdown("---") # Separator

    with tabs[6]:
        file_upload_panel()

    with tabs[7]:
        notifications_panel()

    with tabs[8]:
        st.header("Export Data")
        st.write("Download CSV reports for various collections")
        export_collection_to_csv(users_collection, "users.csv")
        export_collection_to_csv(history_collection, "history.csv")
        export_collection_to_csv(reviews_collection, "reviews.csv")
        export_collection_to_csv(requests_collection, "requests.csv")
        export_collection_to_csv(files_collection, "files.csv")
        export_collection_to_csv(logs_collection, "logs.csv")


        # Simple PDF generation if available
        REPORTLAB_AVAILABLE = False # Assuming reportlab is not installed by default
        try:
            from reportlab.lib.pagesizes import letter
            from reportlab.pdfgen import canvas
            import io
            REPORTLAB_AVAILABLE = True
        except ImportError:
            pass # reportlab is not installed


        if REPORTLAB_AVAILABLE:
            st.subheader("PDF Exports (Requires reportlab)")
            if st.button("Download Users Report (PDF)", key="export_users_pdf"):
                buf = io.BytesIO()
                c = canvas.Canvas(buf, pagesize=letter)
                width, height = letter # Get dimensions

                c.drawString(100, height - 50, "CodeGenie Users Report")
                c.drawString(100, height - 65, f"Generated: {datetime.datetime.utcnow().strftime('%Y-%m-%d %H:%M:%S')} UTC")

                users = list(users_collection.find({}).sort("created_at",-1).limit(500))
                if users:
                    y_position = height - 100
                    for u in users:
                        if y_position < 50: # Check if we need a new page
                            c.showPage()
                            y_position = height - 50
                            c.drawString(100, y_position, "CodeGenie Users Report (cont.)")
                            y_position -= 20

                        c.drawString(100, y_position, f"Username: {u.get('username', 'N/A')}")
                        c.drawString(300, y_position, f"Role: {u.get('role', 'user')}")
                        c.drawString(450, y_position, f"Joined: {u.get('created_at').strftime('%Y-%m-%d') if u.get('created_at') else 'N/A'}")
                        y_position -= 15 # Move down for the next user

                c.save()
                buf.seek(0)
                st.download_button("Download PDF", buf, file_name=f"users_report_{datetime.datetime.utcnow().strftime('%Y%m%d')}.pdf", mime="application/pdf")
                log_action("export_users_pdf", username=st.session_state.username)

        else:
            st.info("`reportlab` not installed: PDF export disabled. Run `!pip install reportlab` in a code cell to enable.")


# --- Initial Admin User Creation (if needed) ---
def create_initial_admin():
    """
    Checks if initial admin credentials are provided in .env and creates the user
    if the 'users' collection is empty.
    """
    # Ensure collections exist before trying to count/insert
    if "users" not in db.list_collection_names():
        # If users collection doesn't exist, create it implicitly on first insert
        pass # pymongo handles creation on first write

    if ADMIN_INITIAL_USER and ADMIN_INITIAL_PASS:
        # Check if ANY user exists
        if users_collection.count_documents({}) == 0:
            # Check if a user with the initial admin username already exists (edge case)
            existing_initial_admin = users_collection.find_one({"username": ADMIN_INITIAL_USER})
            if not existing_initial_admin:
                st.info(f"Creating initial admin user: {ADMIN_INITIAL_USER}")
                try:
                    hashed = bcrypt.hashpw(ADMIN_INITIAL_PASS.encode('utf-8'), bcrypt.gensalt())
                    users_collection.insert_one({
                        "username": ADMIN_INITIAL_USER,
                        "password": hashed,
                        "role": "admin",
                        "created_at": datetime.datetime.utcnow(),
                        "notes": "Initial system admin created on first run"
                    })
                    st.success(f"Initial admin user '{ADMIN_INITIAL_USER}' created. You can now log in.")
                    log_action("initial_admin_created", username="system", meta={"username": ADMIN_INITIAL_USER})
                except Exception as e:
                    st.error(f"Failed to create initial admin user: {e}")
                    log_action("initial_admin_creation_failed", username="system", status="failed", meta={"error": str(e)})
            else:
                 # This case happens if the collection was empty, initial admin was tried, failed partially,
                 # but the user document was created without the role/password logic completing.
                 # Or if the user manually created a user with that email before the script ran.
                 st.warning(f"Initial admin user '{ADMIN_INITIAL_USER}' already exists. Skipping auto-creation.")
        # else:
             # print("Users collection is not empty. Skipping initial admin auto-creation.") # For debugging
# -----------------------------
# 🔆 Main Navigation
# -----------------------------
def main_interface():
    apply_neon_css()  # Apply theme
    st.sidebar.title("⚡ CodeGenie")

    # Use session state role to conditionally show Admin Panel
    role = st.session_state.get("role", "user")

    if role == 'user':
        menu_options = ["💻 Workspace", "⟲ History", "🧠 General Chat", "👤 Profile"]
    elif role == 'admin':
        menu_options = ["👑 Admin Panel", "👤 Profile"]
    else:
        menu_options = ["💻 Workspace", "👤 Profile"]

    page = st.sidebar.radio("", menu_options)

    if st.sidebar.button("Logout"):
        st.session_state.clear()
        st.session_state.page = "login"  # Redirect to login
        st.rerun()

    # --- Page Mapping ---
    if page == "💻 Workspace":
        workspace_page()
    elif page == "⟲ History":
        history_page()
    elif page == "🧠 General Chat":
        general_chat_page()
    elif page == "👤 Profile":
        profile_page()
    elif page == "👑 Admin Panel" and role == "admin":
        admin_panel()
    elif page == "👑 Admin Panel" and role != "admin":
        # Should not happen due to menu options, but good safeguard
        st.error("Access Denied: You do not have admin privileges.")


# -----------------------------
# 🚀 App Flow
# -----------------------------
st.set_page_config(page_title="CodeGenie", page_icon="⚡", layout="wide")

# Initialize session state variables if they don't exist
if "page" not in st.session_state:
    st.session_state.page = "login"
if "jwt_token" not in st.session_state:
    st.session_state.jwt_token = None
if "username" not in st.session_state:
    st.session_state.username = None
if "role" not in st.session_state: # Store user role in session state
    st.session_state.role = None
if "otp_stage" not in st.session_state:
    st.session_state.otp_stage = None
if "otp_user" not in st.session_state:
    st.session_state.otp_user = None


# --- Run initial admin creation check on app load ---
# This will only run when the script is first executed or reran fully
create_initial_admin()


# --- Main App Routing ---
if st.session_state.page == "login":
    login_signup_page()
elif st.session_state.page == "otp":
    if "otp_stage" not in st.session_state or st.session_state.get("otp_stage") is None:
        st.session_state['otp_stage'] = "request"
    password_reset_page()
else: # Assume logged in, verify token
    decoded = decode_token(st.session_state.get("jwt_token"))
    if decoded:
        # Ensure session state is updated from token in case of rerun
        st.session_state.username = decoded.get("username")
        st.session_state.role = decoded.get("role", "user") # Get role from token
        main_interface()
    else:
        # If token is invalid or expired
        st.warning("Your session has expired. Please log in again.")
        st.session_state.clear() # Clear all session state on expired token
        st.session_state.page = "login" # Redirect to login
        st.rerun()

In [ ]:
!pip install --no-input pyngrok

from pyngrok import ngrok

In [ ]:
!kill -9 $(lsof -t -i:8501)

!streamlit run app.py --server.port 8501 &>/content/logs.txt &

from pyngrok import ngrok

# ✅ Replace with your *new* Ngrok authtoken
ngrok.set_auth_token("35Nu4IJrt70FCjoZ1j9SeRVgZAi_5ZnLjymuDLdWJuJzrkUvT") # <-- Replace this with your valid ngrok authtoken

# 🚀 Start the tunnel
public_url = ngrok.connect(8501)
print("🚀 Your public URL:", public_url.public_url)

In [ ]:
# ngrok.kill()